<a href="https://colab.research.google.com/github/rpp5069-cpu/job-pipeline/blob/main/CS_MBA_AI_USAJobs_Jooble_Master_Combiner_v3_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS / MBA / Data Analytics / AI
## USAJobs + Jooble — Production Collection & O\\*NET 24.2 Competency Mapper
### Google Colab + Google Drive | Three Automation Backends
**Penn State Great Valley | D.Eng. Research — AI in the Workforce & Labor Market**
--## Pipeline flow
```
USAJobs API ──┐
├──► Section 4–5: Live Deep Fetch
Jooble API ──┘
│
Section 6: Score + Merge + Dedup
│
Section 3B: O*NET 24.2 (35 skills)
│
Section 7: Save to Google Drive
Research/JobPostings/
├── outputs/ MASTER_YYYYMMDD_HHMM.{json,csv,xlsx}
└── archive/ (prior runs)
```
**Usage:** `Runtime → Run all`. Drive auth dialog appears once; everything runs
automatically.
| Automation option | Where configured |
|---|---|

| **A — Google Apps Script** (simplest) | Section 8-A markdown — copy-paste JS into
script.google.com |
| **B — GitHub Actions** (most reliable free) | Section 8-B markdown — copy-paste YAML
into repo |
| **C — GCP Cloud Scheduler + Cloud Run** (production) | Section 8-C markdown — gcloud
commands |
**API keys required** (add once in Colab Secrets

🔑)

| Secret | Source |
|---|---|
| `USAJOBS_EMAIL` | Registered email at developer.usajobs.gov |
| `USAJOBS_KEY` | API key from the same registration |
| `JOOBLE_KEY` | Free key at jooble.org/api (500 req/day) |

In [5]:
!pip install -q pandas openpyxl requests beautifulsoup4 tqdm
print("Packages ready.")

Packages ready.


In [6]:
import re, json, os, hashlib, time, shutil, warnings
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
from collections import Counter
from itertools import chain
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')
# ── Credential loader: works in Colab AND in Papermill / GitHub Actions ───────
def _get_secret(key):
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception:
        return os.environ.get(key, '')
USAJOBS_EMAIL = _get_secret('USAJOBS_EMAIL')
USAJOBS_KEY = _get_secret('USAJOBS_KEY')
JOOBLE_KEY = '4ad82ea7-0118-4f3a-8825-20ed48373e17'
CURRENT_EPOCH = datetime(2026, 1, 1)
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M')
ONET_VERSION = 'O*NET 24.2'
MASTER_JSON = f'MASTER_{RUN_TS}.json'
MASTER_CSV = f'MASTER_{RUN_TS}.csv'
MASTER_XLSX = f'MASTER_{RUN_TS}.xlsx'


print(f"Imports complete. Run: {RUN_TS} | Epoch: {CURRENT_EPOCH.date()}")
print(f" USAJOBS_EMAIL : {'✓' if USAJOBS_EMAIL else '✗ MISSING'}")
print(f" USAJOBS_KEY : {'✓' if USAJOBS_KEY else '✗ MISSING'}")
print(f" JOOBLE_KEY : {'✓' if JOOBLE_KEY else '✗ MISSING'}")

Imports complete. Run: 20260627_0249 | Epoch: 2026-01-01
 USAJOBS_EMAIL : ✓
 USAJOBS_KEY : ✓
 JOOBLE_KEY : ✓


--## Section 1 — Configuration
Expand or trim keyword lists to adjust coverage. Default settings target maximum
volume within free-tier quotas:
| Source | Keywords | Pages | Results/page | Max raw |
|--------|----------|-------|-------------|---------|
| USAJobs | 37 | 10 | 25 | ~9,250 |
| Jooble | 33 | 5 | ~20 | ~3,300 |

In [7]:
# --- USAJobs keyword map --- 4 domains
USAJOBS_QUERIES = {
'CS': [
'software engineer', 'computer scientist',
'information technology specialist', 'cybersecurity analyst',
'cloud engineer', 'devops engineer', 'systems administrator',
'network engineer', 'platform engineer', 'site reliability engineer',
'solutions architect', 'security engineer',
],
'MBA': [
'management analyst', 'program analyst',
'operations research analyst', 'financial analyst',
'budget analyst', 'management and program analyst',
'administrative officer', 'product manager',
'strategy analyst', 'corporate strategy',
],
'DATA_ANALYTICS': [
'data analyst', 'statistician', 'data scientist',
'business intelligence analyst', 'quantitative analyst',
'data engineer', 'reporting analyst', 'analytics engineer',
'senior data analyst', 'data visualization analyst',
],
'AI': [
'artificial intelligence', 'machine learning engineer',
'natural language processing', 'computer vision engineer',
'applied scientist', 'MLOps engineer', 'deep learning engineer',
'AI researcher', 'generative AI engineer', 'NLP engineer',
'LLM engineer', 'AI product manager', 'reinforcement learning',
'research scientist', 'data scientist machine learning',
],
}


# --- Jooble keyword map --- 4 domains
JOOBLE_QUERIES = {
'CS': [
'software engineer', 'DevOps engineer', 'cloud engineer',
'backend developer', 'full stack developer', 'cybersecurity analyst',
'systems architect', 'platform engineer', 'solutions architect',
'security engineer', 'network engineer',
],
'MBA': [
'strategy consultant', 'product manager', 'business analyst',
'operations manager', 'management consultant', 'financial analyst',
'corporate strategy', 'program manager', 'management analyst',
],
'DATA_ANALYTICS': [
'data analyst', 'business intelligence analyst', 'analytics engineer',
'data engineer', 'BI developer', 'quantitative analyst',
'data visualization analyst', 'reporting analyst', 'statistician',
],
'AI': [
'machine learning engineer', 'AI researcher', 'NLP engineer',
'MLOps engineer', 'generative AI engineer', 'applied scientist',
'deep learning engineer', 'LLM engineer', 'computer vision engineer',
'reinforcement learning engineer', 'AI product manager',
'prompt engineer', 'data scientist',
],
}
# --- Pagination ---
USAJOBS_PAGE_SIZE = 25
# max 500; 25 is well within rate limits
USAJOBS_MAX_PAGES = 10
# 10 × 25 = 250 per keyword
JOOBLE_MAX_PAGES = 5
# ~20 results per page × 5 = 100 per keyword
# --- Rate limiting (seconds between consecutive API calls) ---
USAJOBS_DELAY = 1.0
JOOBLE_DELAY = 0.8
# --- Minimum relevance score to keep a record ---
MIN_SCORE = 2
usa_kw = sum(len(v) for v in USAJOBS_QUERIES.values())
jbl_kw = sum(len(v) for v in JOOBLE_QUERIES.values())
print(f"Configuration ready.")
print(f" USAJobs : {usa_kw} keywords × {USAJOBS_MAX_PAGES} pages ×\n{USAJOBS_PAGE_SIZE} = up to {usa_kw*USAJOBS_MAX_PAGES*USAJOBS_PAGE_SIZE:,} raw")
print(f" Jooble\n: {jbl_kw} keywords × {JOOBLE_MAX_PAGES} pages × ~20\n= up to\n{jbl_kw*JOOBLE_MAX_PAGES*20:,} raw")

Configuration ready.
 USAJobs : 47 keywords × 10 pages ×
25 = up to 11,750 raw
 Jooble
: 42 keywords × 5 pages × ~20
= up to
4,200 raw


---

## Section 2 — Google Drive Mount & Folder Setup
Drive is mounted at `/content/drive`. All output files are copied there directly
via `shutil.copy2` — no extra API credentials needed.
Change `DRIVE_BASE` if your folder is located elsewhere in My Drive.

In [8]:
import os, shutil
import pandas as pd

# --- Environment-Aware Drive Setup ---
try:
    from google.colab import drive as colab_drive
    colab_drive.mount('/content/drive', force_remount=False)
    print("Google Drive mounted at /content/drive")
    IS_COLAB = True
except ImportError:
    print("Running outside Colab (Headless/GitHub Actions) - Skipping Drive Mount")
    IS_COLAB = False

# --- Papermill Parameters Cell ---
# The 'parameters' tag MUST be added to this cell's metadata.
# Default path for Colab; GitHub Actions will override this via the workflow YAML.
DRIVE_BASE = globals().get('DRIVE_BASE', '/content/drive/MyDrive/Research/JobPostings')

DRIVE_OUTPUTS = os.path.join(DRIVE_BASE, 'outputs')
DRIVE_ARCHIVE = os.path.join(DRIVE_BASE, 'archive')
DRIVE_LOGS = os.path.join(DRIVE_BASE, 'logs')
LOCAL_WORK = './job_pipeline_work'

for folder in [DRIVE_OUTPUTS, DRIVE_ARCHIVE, DRIVE_LOGS, LOCAL_WORK]:
    os.makedirs(folder, exist_ok=True)

print(f"outputs : {DRIVE_OUTPUTS}")
print(f"archive : {DRIVE_ARCHIVE}")

def save_to_drive(local_path: str, drive_dir: str, fname: str) -> str:
    """Copy a local file to the destination directory."""
    dst = os.path.join(drive_dir, fname)
    shutil.copy2(local_path, dst)
    kb = os.path.getsize(dst) / 1024
    print(f"{fname} ({kb:.1f} KB) → {drive_dir} ✅")
    return dst

def write_run_log(df: pd.DataFrame) -> None:
    log = os.path.join(DRIVE_LOGS, 'run_log.tsv')
    needs_header = not os.path.exists(log)
    RUN_TS = pd.Timestamp.now().strftime('%Y%m%d_%H%M')
    with open(log, 'a', encoding='utf-8') as fh:
        if needs_header:
            fh.write('run_ts\ttotal\tcurrent\thistoric\tUSAJobs\tJooble\thigh\tmedium\tlow\n')
        fh.write('\t'.join([
            RUN_TS, str(len(df)),
            str((df.era=='current').sum()), str((df.era=='historical').sum()),
            str((df.source=='USAJobs').sum()), str((df.source=='Jooble').sum()),
            str((df.relevance_tier=='HIGH').sum()),
            str((df.relevance_tier=='MEDIUM').sum()),
            str((df.relevance_tier=='LOW').sum()),
        ]) + '\n')

Mounted at /content/drive
Google Drive mounted at /content/drive
outputs : /content/drive/MyDrive/Research/JobPostings/outputs
archive : /content/drive/MyDrive/Research/JobPostings/archive


In [9]:
import json
import os
from google.colab import _message

def add_parameters_tag_to_s2():
    """Programmatically adds the 'parameters' tag to the s2-drive cell."""
    # 1. Identify the notebook path (standard Colab location)
    # Note: This works by modifying the current session's metadata
    print("Applying 'parameters' tag to cell 's2-drive'...")

    try:
        # We use a Colab internal command to update the metadata of cell 's2-drive'
        _message.blocking_request(
            'set_cell_metadata',
            request={'cellId': 's2-drive', 'metadata': {'tags': ['parameters']}},
            timeout_sec=5
        )
        print("✅ Success! The 'parameters' tag has been added to the 's2-drive' cell.")
        print("ACTION REQUIRED: Press Ctrl+S (or Cmd+S) NOW to save this change to the file.")
    except Exception as e:
        print(f"❌ Could not apply tag automatically: {e}")
        print("Please try the 'Command Palette' (Ctrl+Shift+P) and search for 'Edit cell metadata' one last time.")

add_parameters_tag_to_s2()

Applying 'parameters' tag to cell 's2-drive'...
✅ Success! The 'parameters' tag has been added to the 's2-drive' cell.
ACTION REQUIRED: Press Ctrl+S (or Cmd+S) NOW to save this change to the file.


--## Section 3A — Domain Relevance Scoring Engine & Utilities
| Function | Purpose |
|---|---|
| `score_relevance()` | 0–12 score; HIGH ≥ 6 / MEDIUM ≥ 4 / LOW ≥ 2 / EXCLUDE < 2 |
| `extract_skills()` | Technical tools & languages (SKILL_TAXONOMY) |
| `infer_domain()` | CS / MBA / DATA_ANALYTICS / AI |
| `classify_era()` | `current` (≥ Jan 2026) or `historical` |
| `make_hash()` | SHA-1 of title+company+location for cross-source dedup |
| `normalize_location()` | Standardize to `City, ST, US` format |

In [10]:
# --- Technical skill taxonomy
SKILL_TAXONOMY = [
'Python','R','SQL','Java','Scala','Go','JavaScript','TypeScript','C++','Rust','MATLAB',
'Julia',
'PyTorch','TensorFlow','Keras','Scikit-learn','XGBoost','LightGBM','Hugging Face','JAX','ONNX','LangChain',
'Machine Learning','Deep Learning','NLP','Computer Vision','Reinforcement Learning','MLOps',
'Feature Engineering','Model Deployment','A/B Testing','Large Language Models',
'Generative AI','RAG','Prompt Engineering','Fine-tuning','RLHF',
'Spark','Kafka','Airflow','dbt','Databricks','Snowflake','BigQuery','Redshift',
'ETL','ELT','Data Warehouse','Data Lake','Delta Lake','Flink',
'Tableau','Power BI','Looker','Qlik','Excel','Statistics','Pandas','NumPy',
'Matplotlib','Seaborn','Plotly',
'AWS','Azure','GCP','Docker','Kubernetes','Terraform','Helm','ArgoCD',
'REST API','GraphQL','CI/CD','Git','Linux','Microservices','Cybersecurity','gRPC',
'Financial Modeling','Six Sigma','Agile','Scrum','Project Management',
'Strategic Planning','Business Intelligence','OKRs','Stakeholder Management',
]


DOMAIN_TITLE_T1 = [
'data analyst','senior data analyst','analytics engineer','business intelligence',
'bi developer','bi analyst','quantitative analyst','data visualization','reporting analyst',
'data engineer','machine learning engineer','ml engineer','applied scientist',
'research scientist','ai engineer','nlp engineer','deep learning engineer',
'computer vision engineer','mlops engineer','data scientist','ai researcher',
'generative ai','llm engineer','natural language processing','reinforcement learning',
'software engineer','software developer','systems architect','cloud engineer',
'devops engineer','platform engineer','site reliability engineer','backend engineer',
'cybersecurity analyst','security engineer','solutions architect','computer scientist',
'network engineer','systems administrator','information technology specialist',
'strategy consultant','management consultant','business analyst','product manager',
'operations manager','financial analyst','management analyst','operations analyst',
'program analyst','budget analyst','management and program analyst','program manager',
'prompt engineer','llm engineer','ai product manager',
]
DOMAIN_TITLE_T2 = [
'engineer','developer','analyst','scientist','architect','consultant','manager',
'director','specialist','python','data','cloud','ai','ml','machine learning',
'analytics','intelligence','research','technology','software','information technology',
]
DOMAIN_DESC_TERMS = [
'machine learning','deep learning','neural network','nlp','natural language processing',
'computer vision','pytorch','tensorflow','scikit-learn','hugging face','llm',
'large language model','generative ai','rag','mlops','python','sql','scala','java',
'data pipeline','etl','data warehouse','spark','kafka','airflow','dbt',
'databricks','snowflake','bigquery','redshift','tableau','power bi','looker',
'aws','azure','gcp','docker','kubernetes','terraform','a/b testing','statistics',
'mba','strategy','financial modeling','agile','scrum','rest api','devops',
'cybersecurity','information security','federal','clearance',
]
DOMAIN_ORG_NAMES = [
'department of defense','dod','national security agency','nsa','nasa',
'national science foundation','department of energy','department of veterans affairs',
'centers for disease control','cdc','department of treasury','irs',
'general services administration','gsa','department of commerce','census bureau',
'department of homeland security','department of state','department of labor',
'google','amazon','meta','apple','microsoft','openai','anthropic','deepmind',
'nvidia','intel','ibm','oracle','salesforce','databricks','snowflake','palantir',
'mckinsey','bcg','bain','deloitte','accenture','pwc','booz allen','leidos','mitre',
'jpmorgan','goldman sachs','bloomberg',
]
def _pat(terms):
    esc = [re.escape(t) for t in sorted(terms, key=len, reverse=True)]
    return re.compile(r'\b(' + '|'.join(esc) + r')\b', re.IGNORECASE)
PAT_T1 = _pat(DOMAIN_TITLE_T1)
PAT_T2 = _pat(DOMAIN_TITLE_T2)
PAT_DD = _pat(DOMAIN_DESC_TERMS)
PAT_OR = _pat(DOMAIN_ORG_NAMES)

def score_relevance(title='', description='', company=''):
    score, matched = 0, []
    tl = str(title).lower()
    dl = str(description).lower()[:3000]
    cl = str(company).lower()
    for pat, pts in [(PAT_T1, 4), (PAT_T2, 2)]:
        h = pat.findall(tl)
        if h: score += min(pts, pts * len(set(h))); matched += list(set(h))
    h = PAT_DD.findall(dl)
    if h: score += min(4, len(set(h))); matched += list(set(h))
    h = PAT_OR.findall(cl)
    if h: score += 2; matched += list(set(h))
    tier = ('HIGH' if score >= 6 else
            'MEDIUM' if score >= 4 else
            'LOW' if score >= 2 else
            'EXCLUDE')
    return score, list(set(matched)), tier

def extract_skills(title='', description=''):
    c = (str(title) + ' ' + str(description)).lower()
    return [s for s in SKILL_TAXONOMY if s.lower() in c]

def infer_domain(title='', description=''):
    t = (str(title) + ' ' + str(description)).lower()
    if any(k in t for k in ['machine learning','deep learning','nlp','neural','pytorch',
    'tensorflow','mlops','applied scientist','llm','generative','computer vision',
    'reinforcement','artificial intelligence']): return 'AI'
    if any(k in t for k in ['data analyst','business intelligence','analytics engineer',
    'tableau','power bi','looker','quantitative analyst','statistician']): return 'DATA_ANALYTICS'
    if any(k in t for k in ['mba','strategy','consultant','operations manager',
    'product manager','financial analyst','management analyst','budget analyst',
    'program analyst','program manager']): return 'MBA'
    return 'CS'

def classify_era(date_str):
    if not date_str: return 'unknown'
    try:
        return ('current' if datetime.fromisoformat(str(date_str)[:10]) >= CURRENT_EPOCH else 'historical')
    except Exception: return 'unknown'

def make_hash(title, company, location):
    key = \
    f'{str(title).lower().strip()}|{str(company).lower().strip()}|{str(location).lower().strip()}'
    return hashlib.sha1(key.encode()).hexdigest()[:12]

def clean_html(text):
    if not text or isinstance(text, float): return ''
    return BeautifulSoup(str(text), 'html.parser').get_text(separator=' ', strip=True)

def normalize_location(raw):
    if not raw: return 'Unknown, US'
    raw = str(raw).strip()
    if ', US' in raw: return raw
    STATE_MAP = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
    'Colorado':'CO','Connecticut':'CT','Delaware':'DE','Florida':'FL','Georgia':'GA',
    'Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA',
    'Kansas':'KS','Kentucky':'KY','Louisiana':'LA','Maine':'ME','Maryland':'MD',
    'Massachusetts':'MA','MA':'MA','Minnesota':'MN','Mississippi':'MS',
    'Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV',
    'New Hampshire':'NH','New Jersey':'NJ','New Mexico':'NM','New York':'NY',
    'North Carolina':'NC','North Dakota':'ND','Ohio':'OH','Oklahoma':'OK',
    'Oregon':'OR','Pennsylvania':'PA','Rhode Island':'RI','South Carolina':'SC',
    'South Dakota':'SD','Tennessee':'TN','Texas':'TX','Utah':'UT',
    'Vermont':'VT','Virginia':'VA','Washington':'WA','West Virginia':'WV',
    'Wisconsin':'WI','Wyoming':'WY','District of Columbia':'DC',
    }
    for full, abbr in STATE_MAP.items():
        if full in raw:
            parts = raw.split(',')
            city = parts[0].strip() if len(parts) > 1 else ''
            return f'{city}, {abbr}, US' if city else f'{abbr}, US'
    if 'remote' in raw.lower(): return 'Remote, US'
    return f'{raw}, US'


print("Scoring engine ready.")

Scoring engine ready.


--## Section 3B — O\\*NET 24.2 Competency Framework (35 skills)
Every posting receives two skill fields:
| Field | Contents |
|---|---|
| `required_skills` | Technical tools extracted from job text (Python, SQL, …) |
| `onet_skills` | Up to 35 O\\*NET 24.2 standardised competency names |
Mapping uses keyword/phrase triggers from the official O\\*NET Content Model.
`map_onet_skills()` is called automatically inside both API parsers.

In [11]:
ONET_SKILLS_MAP = {
# ── BASIC SKILLS ──────────────────────────────────────────────────────────
"Reading Comprehension": ["documentation","technical document","specification",
"interpret report","review document","written material","policy document",
"literature review","parse","technical manual"],
"Active Listening": ["active listening","stakeholder","requirements gathering",
"client meetings","user interview","discovery session","elicit requirements"],
"Writing": ["writing","written communication","technical writing","report",
"proposal","white paper","draft","document findings","publish"],
"Speaking": ["presentation","verbal communication","briefing","communicate ideas",
"public speaking","articulate","cross-functional communication"],
"Mathematics":
["math","mathematics","statistics","quantitative","calculus","algebra",
"numerical analysis","probability","linear algebra","regression","optimization"],
"Science": ["scientific","research","experiment","hypothesis","empirical",
"research methodology","scientific method","data collection","study design"],
"Critical Thinking": ["critical thinking","analytical","analysis","evaluate","assess",
"problem-solving","logical","reasoning","judgment","root cause","synthesize",
"evidence-based"],
"Active Learning": ["active learning","continuous learning","self-directed",
"upskill","reskill","adapt","emerging technology","stay current",
"professional development","evolving landscape"],
"Learning Strategies": ["learning strategies","training program","curriculum",
"instructional design","e-learning","onboarding plan","coaching","workshop"],
"Monitoring": ["monitor","track","KPI","metrics","performance monitoring",
"dashboard","alerting","observability","logging","SLA","telemetry"],
# ── SOCIAL SKILLS ─────────────────────────────────────────────────────────
"Social Perceptiveness": ["social perceptiveness","emotional intelligence","empathy",
"interpersonal","team dynamics","cultural awareness","relationship building"],
"Coordination": ["coordination","coordinate","cross-functional","collaboration",
"teamwork","liaison","partner with","multi-team","synchronize"],
"Persuasion": ["persuasion","influence","sales","convince","advocate",
"change management","buy-in","executive presence","drive adoption"],
"Negotiation": ["negotiation","negotiate","contract","vendor management",
"procurement","SLA negotiation","partnership agreement"],
"Instructing": ["instruct","mentor","coach","teach","train","knowledge transfer",
"pair programming","onboard","technical guidance","SME"],
"Service Orientation": ["customer service","client-facing","user support","help desk",
"service delivery","customer success","service level","support ticket"],
# ── COMPLEX PROBLEM SOLVING ────────────────────────────────────────────────
"Complex Problem Solving": ["complex problem","problem solving","debug","troubleshoot",
"root cause analysis","issue resolution","incident response","triage",
"architecture challenge","unstructured problem","ambiguity"],
# ── TECHNICAL SKILLS ──────────────────────────────────────────────────────
"Operations Analysis": ["operations analysis","business analysis","requirements analysis",
"workflow analysis","process improvement","gap analysis","cost-benefit",
"functional requirements","business requirements"],
"Technology Design": ["technology design","system design","architecture","solution design",
"technical design","UI/UX","API design","data modeling","schema design",
"design pattern","technical specification"],
"Equipment Selection": ["tool selection","hardware","infrastructure",
"tool evaluation","vendor evaluation","technology selection","build vs buy"],
"Installation": ["install","deploy","configure","setup","provisioning",
"implementation","rollout","release","CI/CD","Terraform","Helm","ArgoCD"],
"Programming": ["programming","coding","software development","software engineering",
"develop","script","Python","Java","SQL","C++","JavaScript","TypeScript",
"Scala","Go","algorithm","data structure","object-oriented","write code"],
"Operations Monitoring": ["system monitoring","infrastructure monitoring",
"network monitoring","application performance","Datadog","Prometheus",
"Grafana","CloudWatch","PagerDuty","incident management","APM"],
"Operation and Control": ["operation and control","automation","DevOps",
"orchestration","Kubernetes","Airflow","workflow automation",
"MLOps","DataOps","pipeline orchestration","job scheduling"],
"Equipment Maintenance": ["maintenance","patching","system upkeep",
"server maintenance","database maintenance","lifecycle management",
"vulnerability patching"],
"Troubleshooting": ["troubleshoot","debug","diagnose","fix bugs","resolve issues",
"issue tracking","defect","log analysis","error handling","postmortem","RCA"],
"Repairing": ["repair","restore","recovery","disaster recovery","failover",
"remediation","rollback","hotfix","business continuity","DR plan"],
"Quality Control Analysis": ["quality control","QA","QC","testing","unit test",
"integration test","code review","validation","test automation","regression",
"acceptance testing","TDD","pytest","Selenium","performance test"],
# ── SYSTEMS SKILLS ────────────────────────────────────────────────────────
"Judgment and Decision Making": ["decision making","judgment","prioritize",
"strategic decision","risk assessment","trade-off","governance",
"risk management","cost-benefit analysis"],
"Systems Analysis": ["systems analysis","enterprise architecture","data architecture",
"integration design","microservices architecture","distributed systems",
"scalability","end-to-end design","full stack","platform architecture"],
"Systems Evaluation": ["benchmark","performance testing","load test","audit",
"proof of concept","A/B testing","model evaluation","ablation","retrospective"],
# ── RESOURCE MANAGEMENT ────────────────────────────────────────────────────
"Time Management": ["time management","deadline","prioritization","scheduling",
"project management","sprint","agile","scrum","milestone","roadmap",
"backlog","Jira","Confluence"],
"Management of Financial Resources": ["budget","financial management","cost management",
"ROI","P&L","revenue","forecast","financial planning","cost optimization",
"fiscal","cost savings"],
"Management of Material Resources": ["resource allocation","inventory","asset management",
"procurement","capacity planning","cloud cost","resource optimization"],
"Management of Personnel Resources": ["people management","team management","hiring",
"performance review","talent management","workforce planning","manage team",
"supervise","org design","team building","recruitment"],
}
ONET_FAMILIES = {
"Basic Skills":
["Reading Comprehension","Active Listening","Writing","Speaking",
"Mathematics","Science","Critical Thinking","Active Learning",
"Learning Strategies","Monitoring"],
"Social Skills":
["Social Perceptiveness","Coordination","Persuasion",
"Negotiation","Instructing","Service Orientation"],
"Complex Problem Solving":["Complex Problem Solving"],
"Technical Skills":
["Operations Analysis","Technology Design","Equipment Selection",
"Installation","Programming","Operations Monitoring",
"Operation and Control","Equipment Maintenance",
"Troubleshooting","Repairing","Quality Control Analysis"],
"Systems Skills":
["Judgment and Decision Making","Systems Analysis","Systems Evaluation"],
"Resource Management":
["Time Management","Management of Financial Resources",
"Management of Material Resources","Management of Personnel Resources"],
}
ONET_SKILL_TO_FAMILY = {sk: fam for fam, sks in ONET_FAMILIES.items() for sk in sks}


def map_onet_skills(title='', description='', existing_skills=None):
    corpus = (str(title) + ' ' + str(description) + ' ' + ' '.join(existing_skills or
    [])).lower()
    return [sk for sk, triggers in ONET_SKILLS_MAP.items()
            if any(t.lower() in corpus for t in triggers)]

def onet_family_profile(onet_skills):
    p = {f: 0 for f in ONET_FAMILIES}
    for sk in onet_skills:
        fam = ONET_SKILL_TO_FAMILY.get(sk)
        if fam: p[fam] += 1
    return p
print(f"O*NET {ONET_VERSION} — {len(ONET_SKILLS_MAP)} skill elements across {len(ONET_FAMILIES)} families.")

O*NET O*NET 24.2 — 35 skill elements across 6 families.


--## Section 4 — USAJobs API: Maximum-Depth Federal Collection
**Endpoint:** `https://data.usajobs.gov/api/search`
**Auth:** `Authorization-Key` header + `User-Agent` (registered email)
**Expected runtime:** ~18–25 min for full 10-page depth across all 37 keywords.

In [12]:
USAJOBS_ENDPOINT = 'https://data.usajobs.gov/api/search'

def usajobs_get_page(keyword, page=1):
    """Fetch one page of USAJobs results with up to 3 retries + exponential
    back-off on 429."""
    if not (USAJOBS_KEY and USAJOBS_EMAIL):
        return []
    headers = {
        'Host': 'data.usajobs.gov',
        'User-Agent': USAJOBS_EMAIL,
        'Authorization-Key': USAJOBS_KEY,
    }
    params = {
        'Keyword': keyword,
        'ResultsPerPage': USAJOBS_PAGE_SIZE,
        'Page': page,
        'CountrySubDivisionCode': 'US',
    }
    for attempt in range(3):
        try:
            r = requests.get(USAJOBS_ENDPOINT, headers=headers, params=params,
                            timeout=30)
            r.raise_for_status()
            return r.json().get('SearchResult', {}).get('SearchResultItems', [])
        except requests.exceptions.HTTPError as e:
            if r.status_code == 429:
                wait = (2 ** attempt) * 10
                print(f'[USAJobs] Rate limited — waiting {wait}s')
                time.sleep(wait)
            else:
                print(f'[USAJobs] HTTP {r.status_code} "{keyword}" p{page}')
                return []
        except Exception as e:
            print(f'[USAJobs] Error attempt {attempt+1}: {e}')
            time.sleep(5)
    return []

def parse_usajobs_item(item, keyword, domain):
    mv = item.get('MatchedObjectDescriptor', {})
    title = mv.get('PositionTitle', '')
    org = mv.get('OrganizationName', 'Federal Agency')
    locs = mv.get('PositionLocation', [])
    loc_raw = locs[0].get('LocationName', '') if locs else ''
    qual = mv.get('QualificationSummary', '')
    ua = mv.get('UserArea', {}).get('Details', {})
    duties = ua.get('MajorDuties', [])
    desc = clean_html(qual + ' ' + (' '.join(duties) if isinstance(duties, list)
                                   else str(duties)))
    pub_date = mv.get('PublicationStartDate', '')[:10]
    close_dt = mv.get('ApplicationCloseDate', '')[:10]
    url = mv.get('PositionURI', '')
    rem = mv.get('PositionRemuneration', [])
    salary = ''
    if rem:
        lo = rem[0].get('MinimumRange', ''); hi = rem[0].get('MaximumRange', '')
        iv = rem[0].get('RateIntervalCode', '')
        salary = f'${lo}-${hi} {iv}'.strip()
    sched = mv.get('PositionSchedule', [])
    job_type = sched[0].get('Name', '') if sched else ''
    grades = mv.get('JobGrade', [])
    grade = ', '.join(g.get('Code', '') for g in grades)
    score, matched, tier = score_relevance(title, desc, org)
    tech = extract_skills(title, desc)
    onet = map_onet_skills(title, desc, tech)
    return {
        'hash_id': make_hash(title, org, loc_raw),
        'job_title': title,
        'company': org,
        'location': normalize_location(loc_raw),
        'required_skills': tech,
        'technical_skills': tech,
        'onet_skills': onet,
        'domain': domain,
        'era': classify_era(pub_date),
        'publication_date': pub_date,
        'application_close': close_dt,
        'salary': salary,
        'job_type': job_type,
        'grade': grade,
        'job_url': url,
        'description': desc[:600],
        'relevance_score': score,
        'matched_terms': '|'.join(matched),
        'relevance_tier': tier,
        'source': 'USAJobs',
        'search_keyword': keyword,
    }

def collect_usajobs(queries=None, max_pages=None, min_score=None):
    queries = queries or USAJOBS_QUERIES
    max_pages = USAJOBS_MAX_PAGES if max_pages is None else max_pages
    min_score = MIN_SCORE if min_score is None else min_score
    if not (USAJOBS_KEY and USAJOBS_EMAIL):
        print('USAJobs credentials missing — skipping.')
        return []
    all_records, seen = [], set()
    for domain, keywords in queries.items():
        for keyword in tqdm(keywords, desc=f'USAJobs [{domain}]'):
            for page in range(1, max_pages + 1):
                items = usajobs_get_page(keyword, page)
                if not items: break
                for item in items:
                    mv = item.get('MatchedObjectDescriptor', {})
                    t = mv.get('PositionTitle', '')
                    org = mv.get('OrganizationName', '')
                    locs = mv.get('PositionLocation', [])
                    loc = locs[0].get('LocationName', '') if locs else ''
                    hid = make_hash(t, org, loc)
                    if hid in seen: continue
                    qs, _, _ = score_relevance(t, '', org)
                    if qs == 0: continue
                    rec = parse_usajobs_item(item, keyword, domain)
                    if rec['relevance_score'] >= min_score:
                        seen.add(hid)
                        all_records.append(rec)
                time.sleep(USAJOBS_DELAY)
    print(f'USAJobs total: {len(all_records)} records')
    return all_records
print("USAJobs collector ready.")

USAJobs collector ready.


In [13]:
print('Collecting from USAJobs...')
print('=' * 60)

try:
    usajobs_records = collect_usajobs()
    if usajobs_records:
        df_usa = pd.DataFrame(usajobs_records)
        print(f'\n Records : {len(df_usa)}')
        print(f' HIGH tier : {(df_usa.relevance_tier=="HIGH").sum()}')
        print(f' Current (>=2026): {(df_usa.era=="current").sum()}')
        print('\nDomain breakdown:')
        print(df_usa.domain.value_counts().to_string())
        all_onet_u = list(chain.from_iterable(df_usa.onet_skills.dropna()))
        print('\nTop 8 O*NET skills (USAJobs):')
        for sk, n in Counter(all_onet_u).most_common(8):
            print(f' {sk:<42}: {n}')
    else:
        df_usa = pd.DataFrame()
        print('No USAJobs results — verify USAJOBS_EMAIL and USAJOBS_KEY in Colab Secrets.')
except Exception as e:
    print(f"An error occurred during USAJobs collection: {e}")
    df_usa = pd.DataFrame() # Ensure df_usa is defined even on error

USAJobs [CS]:   0%|          | 0/12 [00:00<?, ?it/s]

USAJobs [MBA]:   0%|          | 0/10 [00:00<?, ?it/s]

USAJobs [DATA_ANALYTICS]:   0%|          | 0/10 [00:00<?, ?it/s]

USAJobs [AI]:   0%|          | 0/15 [00:00<?, ?it/s]

USAJobs total: 1098 records

 Records : 1098
 HIGH tier : 448
 Current (>=2026): 1057

Domain breakdown:
domain
MBA               497
CS                421
DATA_ANALYTICS    123
AI                 57

Top 8 O*NET skills (USAJobs):
 Programming                               : 1075
 Critical Thinking                         : 975
 Instructing                               : 966
 Monitoring                                : 817
 Writing                                   : 755
 Coordination                              : 552
 Installation                              : 481
 Science                                   : 473


--## Section 5 — Jooble API: Maximum-Depth Commercial Collection
**Endpoint:** `https://jooble.org/api/{key}` (POST)
**Quota:** 500 requests/day free tier
**Expected runtime:** ~8–12 min for full 5-page depth across all 33 keywords.

In [14]:
def jooble_get_page(keyword, page=1, location='United States'):
    if not JOOBLE_KEY: return []
    url = f'https://jooble.org/api/{JOOBLE_KEY}'
    payload = {'keywords': keyword, 'location': location, 'page': page}
    for attempt in range(3):
        try:
            r = requests.post(url, json=payload, timeout=30)
            r.raise_for_status()
            return r.json().get('jobs', [])
        except requests.exceptions.HTTPError:
            print(f'[Jooble] HTTP {r.status_code} "{keyword}" p{page}')
            return []
        except Exception as e:
            print(f'[Jooble] Error attempt {attempt+1}: {e}')
            time.sleep(5)
    return []

def parse_jooble_job(job, keyword, domain):
    title = job.get('title', '')
    company = job.get('company', 'Unknown Company')

    loc_raw = job.get('location', '')
    snippet = clean_html(job.get('snippet', '') or job.get('description', ''))
    pub_date = str(job.get('updated', ''))[:10]
    score, matched, tier = score_relevance(title, snippet, company)
    tech = extract_skills(title, snippet)
    onet = map_onet_skills(title, snippet, tech)
    return {
        'hash_id': make_hash(title, company, loc_raw),
        'job_title': title,
        'company': company,
        'location': normalize_location(loc_raw),
        'required_skills': tech,
        'technical_skills': tech,
        'onet_skills': onet,
        'domain': domain,
        'era': classify_era(pub_date),
        'publication_date': pub_date,
        'application_close': '',
        'salary': job.get('salary', ''),
        'job_type': job.get('type', ''),
        'grade': '',
        'job_url': job.get('link', ''),
        'description': snippet[:600],
        'relevance_score': score,
        'matched_terms': '|'.join(matched),
        'relevance_tier': tier,
        'source': 'Jooble',
        'search_keyword': keyword,
    }

def collect_jooble(queries=None, max_pages=None, min_score=None):
    queries = queries or JOOBLE_QUERIES
    max_pages = JOOBLE_MAX_PAGES if max_pages is None else max_pages
    min_score = MIN_SCORE if min_score is None else min_score
    if not JOOBLE_KEY:
        print('JOOBLE_KEY missing — skipping.')
        return []
    all_records, seen_fp = [], set()
    for domain, keywords in queries.items():
        for keyword in tqdm(keywords, desc=f'Jooble [{domain}]'):
            for page in range(1, max_pages + 1):
                jobs = jooble_get_page(keyword, page=page)
                if not jobs: break
                for job in jobs:
                    fp = f"{job.get('title','').lower()}|{job.get('company','').lower()}|{job.get('location','').lower()}"
                    if fp in seen_fp: continue
                    seen_fp.add(fp)

                    rec = parse_jooble_job(job, keyword, domain)
                    if rec['relevance_score'] >= min_score:
                        all_records.append(rec)
                time.sleep(JOOBLE_DELAY)
    print(f'Jooble total: {len(all_records)} records')
    return all_records
print("Jooble collector ready.")

Jooble collector ready.


In [15]:
print('Collecting from Jooble...')
print('=' * 60)
jooble_records = collect_jooble()
if jooble_records:
    df_jooble = pd.DataFrame(jooble_records)
    print(f'\n Records : {len(df_jooble)}')
    print(f' HIGH tier : {(df_jooble.relevance_tier=="HIGH").sum()}')
    print(f' Current (>=2026): {(df_jooble.era=="current").sum()}')
    print('\nDomain breakdown:')
    print(df_jooble.domain.value_counts().to_string())
    all_onet_j = list(chain.from_iterable(df_jooble.onet_skills.dropna()))
    print('\nTop 8 O*NET skills (Jooble):')
    for sk, n in Counter(all_onet_j).most_common(8):
        print(f' {sk:<42}: {n}')
else:
    df_jooble = pd.DataFrame()
    print('No Jooble results — verify JOOBLE_KEY in Colab Secrets.')

Jooble [CS]:   0%|          | 0/11 [00:00<?, ?it/s]

Jooble [MBA]:   0%|          | 0/9 [00:00<?, ?it/s]

Jooble [DATA_ANALYTICS]:   0%|          | 0/9 [00:00<?, ?it/s]

Jooble [AI]:   0%|          | 0/13 [00:00<?, ?it/s]

Jooble total: 5783 records

 Records : 5783
 HIGH tier : 3968
 Current (>=2026): 5778

Domain breakdown:
domain
AI                1795
CS                1613
MBA               1241
DATA_ANALYTICS    1134

Top 8 O*NET skills (Jooble):
 Programming                               : 2736
 Operation and Control                     : 659
 Critical Thinking                         : 590
 Installation                              : 562
 Writing                                   : 507
 Science                                   : 472
 Monitoring                                : 467
 Equipment Selection                       : 449


In [16]:
def build_master_json(df: pd.DataFrame) -> dict:
    collected_at = datetime.utcnow().isoformat() + 'Z'
    postings = []
    for _, row in df.iterrows():
        tech = row.get('required_skills', [])
        onet = row.get('onet_skills', [])

        # Ensure tech and onet are lists, parsing if they are strings
        if isinstance(tech, str):
            try:
                tech = json.loads(tech)
            except:
                tech = [s.strip() for s in tech.strip('[]').split(',') if s.strip()]
        if not isinstance(tech, list):
            tech = []

        if isinstance(onet, str):
            try:
                onet = json.loads(onet)
            except:
                onet = [s.strip() for s in onet.strip('[]').split(',') if s.strip()]
        if not isinstance(onet, list):
            onet = []

        postings.append({
            'id': int(row.get('id', 0)),
            'title': str(row.get('job_title', '')),
            'company': str(row.get('company', '')),
            'location': str(row.get('location', '')),
            'required_skills': tech,
            'technical_skills': tech,
            'onet_skills': onet,
            'onet_family_profile': onet_family_profile(onet),
            'domain': str(row.get('domain', '')),
            'era': str(row.get('era', '')),
            'posted_date': str(row.get('publication_date', '')),
            'application_close': str(row.get('application_close', '')),
            'salary': str(row.get('salary', '')),
            'job_type': str(row.get('job_type', '')),
            'grade': str(row.get('grade', '')),
            'job_url': str(row.get('job_url', '')),
            'description': str(row.get('description', '')),
            'source': str(row.get('source', '')),
            'search_keyword': str(row.get('search_keyword', '')),
            'relevance_score': int(row.get('relevance_score', 0)),
            'matched_terms': str(row.get('matched_terms', '')),
            'relevance_tier': str(row.get('relevance_tier', '')),
            'hash_id': str(row.get('hash_id', '')),
            'collected_at': collected_at,
        })
    all_onet = list(chain.from_iterable(p['onet_skills'] for p in postings))
    onet_freq = dict(Counter(all_onet).most_common())
    fam_totals = {f: sum(onet_freq.get(sk, 0) for sk in sks)
                  for f, sks in ONET_FAMILIES.items()}
    return {
        'metadata': {
            'title': 'CS / MBA / Data Analytics / AI — MASTER Job Postings',
            'researcher': 'Rickey Patel — Penn State Great Valley D.Eng.',
            'generated_at': collected_at,
            'run_timestamp': RUN_TS,
            'total_records': len(postings),
            'current_records': sum(1 for p in postings if p['era'] == 'current'),
            'historical_records': sum(1 for p in postings if p['era'] == 'historical'),
            'current_epoch': CURRENT_EPOCH.isoformat(),
            'domains': ['CS', 'MBA', 'DATA_ANALYTICS', 'AI'],
            'sources': ['USAJobs', 'Jooble'],
            'schema_version': '3.1',
            'onet_version': ONET_VERSION,
            'onet_top_skills': dict(Counter(all_onet).most_common(15)),
            'onet_family_totals': fam_totals,
            'merge_compatible': True,
        },
        'postings': postings,
    }


frames = [df for df in [
    df_usa if 'df_usa' in dir() and not df_usa.empty else None,
    df_jooble if 'df_jooble' in dir() and not df_jooble.empty else None,
] if df is not None]
if frames:
    df_all = pd.concat(frames, ignore_index=True)
    raw_n = len(df_all)
    df_all = (df_all
              .sort_values('relevance_score', ascending=False)
              .drop_duplicates(subset='hash_id', keep='first')
              .reset_index(drop=True))
    df_all.insert(0, 'id', range(1, len(df_all) + 1))
    master = build_master_json(df_all)
    print(f"Merged: {raw_n} raw → {len(df_all)} unique ({raw_n - len(df_all)} duplicates removed)")
    print(f" Current (≥2026) : {(df_all.era=='current').sum()}")
    print(f" Historical      : {(df_all.era=='historical').sum()}")
    print(f" HIGH tier       : {(df_all.relevance_tier=='HIGH').sum()}")
    print(f" MEDIUM tier     : {(df_all.relevance_tier=='MEDIUM').sum()}")
    print(f" Mean score      : {df_all.relevance_score.mean():.2f}")
    print('\nDomain breakdown:')
    print(df_all.domain.value_counts().to_string())
    print('\nO*NET top 10 (merged):')
    for sk, n in list(master['metadata']['onet_top_skills'].items())[:10]:
        print(f' {sk:<42}: {n:>5} [{ONET_SKILL_TO_FAMILY.get(sk,"")}]')
else:
    print('No data — run Sections 4 and 5 first.')

Merged: 6881 raw → 6881 unique (0 duplicates removed)
 Current (≥2026) : 6835
 Historical      : 46
 HIGH tier       : 4416
 MEDIUM tier     : 310
 Mean score      : 5.26

Domain breakdown:
domain
CS                2034
AI                1852
MBA               1738
DATA_ANALYTICS    1257

O*NET top 10 (merged):
 Programming                               :  3811 [Technical Skills]
 Critical Thinking                         :  1565 [Basic Skills]
 Instructing                               :  1390 [Social Skills]
 Monitoring                                :  1284 [Basic Skills]
 Writing                                   :  1262 [Basic Skills]
 Installation                              :  1043 [Technical Skills]
 Science                                   :   945 [Basic Skills]
 Coordination                              :   848 [Social Skills]
 Operation and Control                     :   739 [Technical Skills]
 Equipment Selection                       :   679 [Technical Skills]


--## Section 7 — Export → Google Drive
Writes three files to `DRIVE_OUTPUTS` using the mounted Drive path (`shutil.copy2`).
No additional OAuth or service-account credentials are needed.
| File | Contents |
|---|---|
| `MASTER_*.json` | Full dataset with `onet_skills` + `onet_family_profile` |
| `MASTER_*.csv` | Flat CSV, skill lists pipe-delimited |
| `MASTER_*.xlsx` | 8 sheets including O\\*NET frequency + family pivot |

In [17]:
if 'master' not in dir() or not master:
    print('No master — run Sections 4–6 first.')
else:
    # ── JSON ──────────────────────────────────────────────────────────────────
    loc_json = os.path.join(LOCAL_WORK, MASTER_JSON)

    with open(loc_json, 'w', encoding='utf-8') as f:
        json.dump(master, f, indent=2, ensure_ascii=False)
    save_to_drive(loc_json, DRIVE_OUTPUTS, MASTER_JSON)
    # ── CSV ───────────────────────────────────────────────────────────────────
    df_csv = df_all.copy()
    for col in ['required_skills', 'technical_skills', 'onet_skills']:
        if col in df_csv.columns:
            df_csv[col] = df_csv[col].apply(
                lambda x: '|'.join(x) if isinstance(x, list) else str(x))
    loc_csv = os.path.join(LOCAL_WORK, MASTER_CSV)
    df_csv.to_csv(loc_csv, index=False, encoding='utf-8-sig')
    save_to_drive(loc_csv, DRIVE_OUTPUTS, MASTER_CSV)
    # ── XLSX — 8 sheets ───────────────────────────────────────────────────────
    df_cur = df_csv[df_csv.era == 'current'].copy()
    df_his = df_csv[df_csv.era == 'historical'].copy()
    df_hm = df_csv[df_csv.relevance_tier.isin(['HIGH', 'MEDIUM'])].copy()
    df_ai = df_csv[df_csv.domain == 'AI'].copy()
    df_csdf = df_csv[df_csv.domain == 'CS'].copy()
    all_onet_flat = list(chain.from_iterable(
        (x if isinstance(x, list) else []) for x in df_all.get('onet_skills', [])))
    onet_freq_df = pd.DataFrame(Counter(all_onet_flat).most_common(),
                                columns=['onet_skill', 'frequency'])
    onet_freq_df['family'] = onet_freq_df['onet_skill'].map(ONET_SKILL_TO_FAMILY)
    onet_freq_df['pct_postings'] = (onet_freq_df['frequency'] / max(len(df_all), 1) *
                                    100).round(1)
    family_df = pd.DataFrame([
        (f, sum(Counter(all_onet_flat).get(sk, 0) for sk in sks))
        for f, sks in ONET_FAMILIES.items()
    ], columns=['onet_family', 'total_mentions'])
    comp = pd.DataFrame({
        'Metric': ['Total','Current','Historical','HIGH','MEDIUM','LOW','Avg Score'],
        'USAJobs': [
            len(df_csv[df_csv.source=='USAJobs']),
            len(df_csv[(df_csv.source=='USAJobs')&(df_csv.era=='current')]),
            len(df_csv[(df_csv.source=='USAJobs')&(df_csv.era=='historical')]),
            len(df_csv[(df_csv.source=='USAJobs')&(df_csv.relevance_tier=='HIGH')]),
            len(df_csv[(df_csv.source=='USAJobs')&(df_csv.relevance_tier=='MEDIUM')]),
            len(df_csv[(df_csv.source=='USAJobs')&(df_csv.relevance_tier=='LOW')]),
            round(df_csv[df_csv.source=='USAJobs'].relevance_score.mean(), 2),
        ],
        'Jooble': [
            len(df_csv[df_csv.source=='Jooble']),
            len(df_csv[(df_csv.source=='Jooble')&(df_csv.era=='current')]),
            len(df_csv[(df_csv.source=='Jooble')&(df_csv.era=='historical')]),
            len(df_csv[(df_csv.source=='Jooble')&(df_csv.relevance_tier=='HIGH')]),
            len(df_csv[(df_csv.source=='Jooble')&(df_csv.relevance_tier=='MEDIUM')]),
            len(df_csv[(df_csv.source=='Jooble')&(df_csv.relevance_tier=='LOW')]),
            round(df_csv[df_csv.source=='Jooble'].relevance_score.mean(), 2),
        ],
        'COMBINED': [
            len(df_csv),
            (df_csv.era=='current').sum(), (df_csv.era=='historical').sum(),
            (df_csv.relevance_tier=='HIGH').sum(),
            (df_csv.relevance_tier=='MEDIUM').sum(),
            (df_csv.relevance_tier=='LOW').sum(),
            round(df_csv.relevance_score.mean(), 2),
        ],
    })
    loc_xlsx = os.path.join(LOCAL_WORK, MASTER_XLSX)
    with pd.ExcelWriter(loc_xlsx, engine='openpyxl') as w:
        df_csv.to_excel(w,
                            sheet_name='All Postings',
                            index=False)
        df_cur.to_excel(w,
                            sheet_name='Current (2026+)',
                            index=False)
        df_his.to_excel(w,
                            sheet_name='Historical (<2026)',
                            index=False)
        df_hm.to_excel(w,
                            sheet_name='HIGH + MEDIUM Tier',
                            index=False)
        df_ai.to_excel(w,
                            sheet_name='AI Domain',
                            index=False)
        df_csdf.to_excel(w,
                            sheet_name='CS Domain',
                            index=False)
        onet_freq_df.to_excel(w, sheet_name='ONET Skill Frequency', index=False)
        family_df.to_excel(w,
                            sheet_name='ONET Family Summary',
                            index=False)
        comp.to_excel(w,
                            sheet_name='Source Comparison',
                            index=False)
    save_to_drive(loc_xlsx, DRIVE_OUTPUTS, MASTER_XLSX)
    # ── Run log ───────────────────────────────────────────────────────────────
    write_run_log(df_all)
    print('\nAll files saved to Drive:')
    for f in sorted(os.listdir(DRIVE_OUTPUTS)):
        kb = os.path.getsize(os.path.join(DRIVE_OUTPUTS, f)) / 1024
        print(f' {f:<65} ({kb:.1f} KB)')
    sample = \
        [{'id':p['id'],'title':p['title'],'required_skills':p['required_skills'][:3],
          'onet_skills':p['onet_skills'][:4]}
         for p in master['postings'] if p['era']=='current'][:3]
    print('\nSample current postings:')
    print(json.dumps(sample, indent=2))

MASTER_20260627_0249.json (10225.9 KB) → /content/drive/MyDrive/Research/JobPostings/outputs ✅
MASTER_20260627_0249.csv (4325.3 KB) → /content/drive/MyDrive/Research/JobPostings/outputs ✅
MASTER_20260627_0249.xlsx (6168.8 KB) → /content/drive/MyDrive/Research/JobPostings/outputs ✅

All files saved to Drive:
 DRY_RUN_TEST_20260620_2057.json                                   (127.4 KB)
 DRY_RUN_TEST_20260621_1159.json                                   (128.3 KB)
 DRY_RUN_TEST_20260623_1840.json                                   (293.3 KB)
 IT_jobs_google_jobs_AllRecords.json                               (12148.2 KB)
 MASTER 20260621 MERGED.json                                       (5030.8 KB)
 MASTER_20260620_1350.csv                                          (1970.4 KB)
 MASTER_20260620_1350.json                                         (3560.5 KB)
 MASTER_20260620_1350.xlsx                                         (1432.5 KB)
 MASTER_20260620_1621.csv                                    

--## Section 8 — Automation Backends
Three production-ready options. Pick one based on your infrastructure.
All three write output files to the same `DRIVE_OUTPUTS` path — no code changes
needed.

--### Option A — Google Apps Script (Simplest)
**How it works:** A JS snippet in your Google account calls the Colab execution API
every Monday at 6 AM. Zero extra accounts, zero cost, entirely inside Google.
**Known limitation:** The Colab execute API is internal/unofficial. If your notebook
isn't found on Drive or the trigger gets an HTTP 4xx, re-save the notebook to Drive
and verify the file ID is correct.
#### Step 1 — Get the notebook file ID
Open the notebook in Colab. The file ID is the long string in the URL:
```
https://colab.research.google.com/drive/1ABC...XYZ
^^^^^^^^^^^ ← copy this
```
#### Step 2 — Create the Apps Script project
1. Go to **[script.google.com](https://script.google.com)** → **New project**
2. Paste the code below; replace `YOUR_NOTEBOOK_FILE_ID`
3. Save → Run → `weeklyJobCollection` (one-time permission grant)
```javascript
// ── Google Apps Script: Weekly Job Collection ────────────────────────────────
const NOTEBOOK_FILE_ID = 'YOUR_NOTEBOOK_FILE_ID'; // ← paste here
const NOTIFY_EMAIL
= Session.getActiveUser().getEmail();
function executeNotebook() {
const token = ScriptApp.getOAuthToken();
const url
=
`https://colab.research.google.com/v2/drives/drive/files/${NOTEBOOK_FILE_ID}:execute`;
const opts = {
method:
'post',
headers:
{ Authorization: `Bearer ${token}`, 'Content-Type':
'application/json' },
payload:
JSON.stringify({ accelerator_type: 'NONE' }),
muteHttpExceptions: true,
};
const resp = UrlFetchApp.fetch(url, opts);
const code = resp.getResponseCode();
Logger.log(`HTTP ${code} — ${resp.getContentText().substring(0, 200)}`);
return code;
}
function weeklyJobCollection() {
Logger.log('Run started: ' + new Date());
const code = executeNotebook();

✅
❌

const subject = code >= 200 && code < 300
? '
Job Pipeline Triggered'
: `
Job Pipeline FAILED (HTTP ${code})`;
const body = code >= 200 && code < 300
? `Pipeline triggered at ${new Date()}. Check Drive/Research/JobPostings/outputs/
in ~35 min.`
: `Trigger failed (HTTP ${code}). Verify notebook file ID and that notebook is
saved in Drive.`;
GmailApp.sendEmail(NOTIFY_EMAIL, subject, body);
}
// Run this once to create the Monday 6 AM trigger programmatically
function createWeeklyTrigger() {
ScriptApp.getProjectTriggers()
.filter(t => t.getHandlerFunction() === 'weeklyJobCollection')
.forEach(t => ScriptApp.deleteTrigger(t));
ScriptApp.newTrigger('weeklyJobCollection')
.timeBased().onWeekDay(ScriptApp.WeekDay.MONDAY).atHour(6).create();
Logger.log('Trigger created: every Monday at 6 AM.');
}
```
#### Step 3 — Add the trigger
Run `createWeeklyTrigger()` once from the Apps Script editor, **or** add it manually:
Clock icon → Add Trigger → `weeklyJobCollection` → Time-driven → Week → Monday → 6–7
AM.
#### Troubleshooting Option A
| Error | Fix |
|---|---|
| `HTTP 404` on execute | Re-save notebook: File → Save a copy in Drive. Re-copy file
ID from new URL. |
| `HTTP 403` / auth failure | Run `weeklyJobCollection` manually first to re-grant
permissions. |
| Files not appearing in Drive | Verify `DRIVE_BASE` path in Section 2 matches your
actual Drive folder. |
| Colab not finishing | Runtime may have been recycled. Re-open notebook, run once
manually, then retry. |
--### Option B — GitHub Actions (Most Reliable Free)
**How it works:** A YAML workflow runs on a GitHub-hosted runner on a cron schedule.
Papermill executes the notebook headlessly. Outputs upload to Drive via a service
account.
**Cost:** Free (2,000 min/month public repo; 500 min/month private). ~30 min per run.

#### Step 1 — Repository setup
```
your-repo/
├── CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb
├── .github/
│
└── workflows/
│
└── weekly_job_collection.yml
└── upload_to_drive.py
```

← this notebook

← paste YAML below
← paste script below

#### Step 2 — Add GitHub repository secrets
`Settings → Secrets and variables → Actions → New repository secret`
| Secret | Value |
|---|---|
| `USAJOBS_EMAIL` | Your USAJobs registered email |
| `USAJOBS_KEY` | USAJobs API key |
| `JOOBLE_KEY` | Jooble API key |
| `GDRIVE_SERVICE_ACCOUNT_JSON` | Contents of GCP service account JSON (see Step 4) |
| `GDRIVE_FOLDER_ID` | ID of the Drive `outputs` folder (from its URL) |
#### Step 3 — `.github/workflows/weekly_job_collection.yml`
```yaml
name: Weekly Job Collection
on:
schedule:
- cron: '0 6 * * 1'
workflow_dispatch:

# Every Monday 06:00 UTC
# Manual trigger from GitHub UI

jobs:
collect:
runs-on: ubuntu-latest
timeout-minutes: 75
steps:
- uses: actions/checkout@v4
- uses: actions/setup-python@v5
with: { python-version: '3.11' }
- name: Install dependencies
run: |
pip install papermill pandas openpyxl requests beautifulsoup4 tqdm \
jupyter nbconvert google-auth google-auth-oauthlib \
google-api-python-client

- name: Execute notebook
env:
USAJOBS_EMAIL: ${{ secrets.USAJOBS_EMAIL }}
USAJOBS_KEY:
${{ secrets.USAJOBS_KEY }}
JOOBLE_KEY:
${{ secrets.JOOBLE_KEY }}
run: |
mkdir -p /tmp/job_outputs /tmp/job_work
papermill CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb \
/tmp/executed_notebook.ipynb \
--no-progress-bar \
-p DRIVE_BASE /tmp/job_outputs
- name: Upload outputs to Google Drive
env:
GDRIVE_SERVICE_ACCOUNT_JSON: ${{ secrets.GDRIVE_SERVICE_ACCOUNT_JSON }}
GDRIVE_FOLDER_ID:
${{ secrets.GDRIVE_FOLDER_ID }}
run: python upload_to_drive.py
- uses: actions/upload-artifact@v4
if: always()
with:
name: executed-notebook-${{ github.run_id }}
path: /tmp/executed_notebook.ipynb
retention-days: 14
```
#### Step 4 — `upload_to_drive.py`
```python
import os, json
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2.service_account import Credentials
sa_info = json.loads(os.environ['GDRIVE_SERVICE_ACCOUNT_JSON'])
creds
= Credentials.from_service_account_info(
sa_info, scopes=['https://www.googleapis.com/auth/drive.file'])
service = build('drive', 'v3', credentials=creds)
folder = os.environ['GDRIVE_FOLDER_ID']
output_dir = '/tmp/job_outputs/outputs'
if not os.path.isdir(output_dir):
output_dir = '/tmp/job_outputs'
for fname in os.listdir(output_dir):
if not fname.startswith('MASTER_'):
continue
fpath = os.path.join(output_dir, fname)
mime = ('application/json' if fname.endswith('.json') else

'text/csv'
if fname.endswith('.csv') else
'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')
meta = {'name': fname, 'parents': [folder]}
media = MediaFileUpload(fpath, mimetype=mime, resumable=True)
f
= service.files().create(body=meta, media_body=media, fields='id').execute()
print(f'Uploaded {fname} → Drive ID {f.get("id")}')
```
#### Step 5 — Create the GCP service account (one time)
```bash
# 1. Create service account
gcloud iam service-accounts create github-pipeline \
--display-name="GitHub Actions Job Pipeline" \
--project=YOUR_PROJECT_ID
# 2. Download JSON key
gcloud iam service-accounts keys create sa_key.json \
--iam-account=github-pipeline@YOUR_PROJECT_ID.iam.gserviceaccount.com
# 3. Share your Drive outputs folder with the service account email:
#
Open Drive folder → Share → paste
github-pipeline@YOUR_PROJECT_ID.iam.gserviceaccount.com
#
→ Editor access
# 4. Paste the contents of sa_key.json into the GDRIVE_SERVICE_ACCOUNT_JSON secret
```
#### Troubleshooting Option B
| Error | Fix |
|---|---|
| `google.colab` import fails | The notebook detects Colab automatically. The
`_get_secret()` helper in Section 0 falls back to `os.environ` on non-Colab runtimes —
no code change needed. |
| Drive upload `insufficientPermissions` | Verify the service account email has
**Editor** access on the Drive folder (not just the file). |
| `papermill` parameter not injected | Ensure Section 2 uses
`globals().get('DRIVE_BASE', …)` — it does by default. |
| Workflow timeout | Increase `timeout-minutes` to 90. Full run is typically 35–45
min. |
--### Option C — GCP Cloud Scheduler + Cloud Run (Production)
**How it works:** A containerised Python script runs on Cloud Run (serverless),
triggered
weekly by Cloud Scheduler. Outputs go to GCS; an optional Drive sync step writes them
to Google Drive as well.

**Cost:** Cloud Run ~$0.20/run · Cloud Scheduler $0.10/job/month → **~$1/month
total**.
#### Architecture
```
Cloud Scheduler (cron) ──HTTP POST──► Cloud Run Job
│
Executes pipeline.py
│
┌────────────┴────────────┐
▼
▼
GCS bucket
Google Drive
gs://PROJECT-outputs/
(optional sync)
```
#### Step 1 — Prepare the standalone script
```bash
# Convert notebook to script (run locally or in Colab)
jupyter nbconvert --to script CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb \
--output pipeline
# Remove all google.colab imports — replace with os.environ.get() (already done
# by the _get_secret() helper in Section 0)
# Remove the colab_drive.mount() call — replace with local /tmp paths
```
Wrap the script in an `if __name__ == '__main__':` guard and change `DRIVE_BASE`
to `/tmp/job_outputs`. Add the GCS upload block below at the end.
#### Step 2 — GCS upload block (add to end of pipeline.py)
```python
from google.cloud import storage as gcs
def upload_to_gcs(local_path, bucket_name, blob_name):
client = gcs.Client()
bucket = client.bucket(bucket_name)
blob
= bucket.blob(blob_name)
blob.upload_from_filename(local_path)
print(f'GCS: gs://{bucket_name}/{blob_name}')
BUCKET = os.environ.get('GCS_BUCKET', 'YOUR_PROJECT_ID-outputs')
for fname in [MASTER_JSON, MASTER_CSV, MASTER_XLSX]:
local = os.path.join('/tmp/job_outputs', fname)
if os.path.exists(local):
upload_to_gcs(local, BUCKET, fname)
```

#### Step 3 — Dockerfile
```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY pipeline.py .
CMD ["python", "pipeline.py"]
```
#### Step 4 — requirements.txt
```
pandas==2.2.2
openpyxl==3.1.2
requests==2.32.3
beautifulsoup4==4.12.3
tqdm==4.66.4
google-cloud-storage==2.17.0
```
#### Step 5 — Deploy
```bash
PROJECT_ID=your-project-id
REGION=us-central1
IMAGE=gcr.io/${PROJECT_ID}/job-pipeline
# Build and push
gcloud builds submit --tag ${IMAGE} --project=${PROJECT_ID}
# Create GCS bucket
gcloud storage buckets create gs://${PROJECT_ID}-outputs \
--location=${REGION} --project=${PROJECT_ID}
# Store secrets in Secret Manager
echo -n "$USAJOBS_EMAIL" | gcloud secrets create USAJOBS_EMAIL --data-file=--project=${PROJECT_ID}
echo -n "$USAJOBS_KEY"
| gcloud secrets create USAJOBS_KEY
--data-file=--project=${PROJECT_ID}
echo -n "$JOOBLE_KEY"
| gcloud secrets create JOOBLE_KEY
--data-file=--project=${PROJECT_ID}
# Create Cloud Run Job
gcloud run jobs create job-pipeline \
--image=${IMAGE} --region=${REGION} \
--memory=2Gi --cpu=1 --task-timeout=3600 \

--set-secrets=USAJOBS_EMAIL=USAJOBS_EMAIL:latest,USAJOBS_KEY=USAJOBS_KEY:latest,JOOBLE
_KEY=JOOBLE_KEY:latest \
--set-env-vars=GCS_BUCKET=${PROJECT_ID}-outputs \
--service-account=pipeline-sa@${PROJECT_ID}.iam.gserviceaccount.com \
--project=${PROJECT_ID}
# Create Cloud Scheduler weekly trigger
gcloud scheduler jobs create http weekly-job-collection \
--location=${REGION} \
--schedule="0 6 * * 1" \
--time-zone="America/New_York" \
--uri="https://${REGION}-run.googleapis.com/apis/run.googleapis.com/v1/namespaces/${PR
OJECT_ID}/jobs/job-pipeline:run" \
--http-method=POST \
--oauth-service-account-email=pipeline-sa@${PROJECT_ID}.iam.gserviceaccount.com \
--project=${PROJECT_ID}
```
#### Step 6 — Grant IAM roles
```bash
SA=pipeline-sa@${PROJECT_ID}.iam.gserviceaccount.com
# Read secrets
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
--member=serviceAccount:${SA} --role=roles/secretmanager.secretAccessor
# Write to GCS bucket
gcloud storage buckets add-iam-policy-binding gs://${PROJECT_ID}-outputs \
--member=serviceAccount:${SA} --role=roles/storage.objectCreator
# Allow Scheduler to invoke Cloud Run
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
--member=serviceAccount:${SA} --role=roles/run.invoker
```
#### Troubleshooting Option C
| Error | Fix |
|---|---|
| `PERMISSION_DENIED` on Secret Manager | Grant `roles/secretmanager.secretAccessor`
to pipeline-sa (Step 6). |
| Container exits immediately | Check Cloud Run logs: `gcloud run jobs executions
describe <exec-id> --region=us-central1 --log-http`. Common cause: missing secrets. |
| GCS upload `403` | Grant `roles/storage.objectCreator` on the specific bucket — not
just project-level. |
| `google.colab` ImportError | Remove `from google.colab import drive` from
pipeline.py and replace `DRIVE_BASE` with `/tmp/job_outputs`. |
| Cloud Scheduler `DEADLINE_EXCEEDED` | The job is still running. Increase

`--task-timeout` to 3600 (1 hr). |

### Step-by-Step Guide: Setting Up GitHub Actions

This guide outlines the process to automate your Colab notebook using GitHub Actions, as described in 'Option B' of your notebook. This method is generally reliable for scheduled, headless execution.

#### Step 1: Repository Setup

First, you need to set up your GitHub repository with the correct file structure. This involves placing your Colab notebook and creating the necessary directories for the GitHub Actions workflow and a helper script.

Your repository should look like this:

```
your-repo/
├── CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb  ← this notebook
├── .github/
│   └── workflows/
│       └── weekly_job_collection.yml               ← paste YAML below
└── upload_to_drive.py                              ← paste script below
```

Ensure your `CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb` file (or whatever your notebook is named) is at the root of your repository.

#### Step 2: Add GitHub Repository Secrets

You'll need to store your API keys and Google Drive service account JSON as GitHub repository secrets. This keeps your sensitive information secure.

1.  Go to your GitHub repository on `github.com`.
2.  Navigate to `Settings` → `Secrets and variables` → `Actions`.
3.  Click `New repository secret` for each of the following:

    | Secret                        | Value                                                               |
    | :---------------------------- | :------------------------------------------------------------------ |
    | `USAJOBS_EMAIL`               | Your USAJobs registered email                                       |
    | `USAJOBS_KEY`                 | USAJobs API key                                                     |
    | `JOOBLE_KEY`                  | Jooble API key                                                      |
    | `GDRIVE_SERVICE_ACCOUNT_JSON` | Contents of GCP service account JSON (from Step 5)                  |
    | `GDRIVE_FOLDER_ID`            | The ID of your Google Drive `outputs` folder (from its URL)         |

#### Step 3: Create `.github/workflows/weekly_job_collection.yml`

This YAML file defines the GitHub Actions workflow. It specifies when the workflow runs, what environment it uses, and the steps to execute your notebook and upload outputs.

Create the file `weekly_job_collection.yml` inside the `.github/workflows/` directory in your repository, and paste the following content:

```yaml
name: Weekly Job Collection
on:
  schedule:
    - cron: '0 6 * * 1' # Every Monday 06:00 UTC
  workflow_dispatch: # Manual trigger from GitHub UI

jobs:
  collect:
    runs-on: ubuntu-latest
    timeout-minutes: 75
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }
      - name: Install dependencies
        run: |
          pip install papermill pandas openpyxl requests beautifulsoup4 tqdm \
            jupyter nbconvert google-auth google-auth-oauthlib \
            google-api-python-client
      - name: Execute notebook
        env:
          USAJOBS_EMAIL: ${{ secrets.USAJOBS_EMAIL }}
          USAJOBS_KEY: ${{ secrets.USAJOBS_KEY }}
          JOOBLE_KEY: ${{ secrets.JOOBLE_KEY }}
        run: |
          mkdir -p /tmp/job_outputs /tmp/job_work
          papermill CS_MBA_AI_USAJobs_Jooble_Master_Combiner.ipynb \
            /tmp/executed_notebook.ipynb \
            --no-progress-bar \
            -p DRIVE_BASE /tmp/job_outputs
      - name: Upload outputs to Google Drive
        env:
          GDRIVE_SERVICE_ACCOUNT_JSON: ${{ secrets.GDRIVE_SERVICE_ACCOUNT_JSON }}
          GDRIVE_FOLDER_ID: ${{ secrets.GDRIVE_FOLDER_ID }}
        run: python upload_to_drive.py
      - uses: actions/upload-artifact@v4
        if: always()
        with:
          name: executed-notebook-${{ github.run_id }}
          path: /tmp/executed_notebook.ipynb
          retention-days: 14
```

#### Step 4: Create `upload_to_drive.py`

This Python script handles uploading the generated output files from the GitHub Actions runner to your Google Drive folder using the service account credentials.

Create the file `upload_to_drive.py` at the root of your repository (the same directory as your notebook) and paste the following content:

```python
import os, json
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.oauth2.service_account import Credentials

sa_info = json.loads(os.environ['GDRIVE_SERVICE_ACCOUNT_JSON'])
creds = Credentials.from_service_account_info(
    sa_info, scopes=['https://www.googleapis.com/auth/drive.file'])
service = build('drive', 'v3', credentials=creds)
folder = os.environ['GDRIVE_FOLDER_ID']

output_dir = '/tmp/job_outputs/outputs'
if not os.path.isdir(output_dir):
    output_dir = '/tmp/job_outputs'

for fname in os.listdir(output_dir):
    if not fname.startswith('MASTER_'):
        continue
    fpath = os.path.join(output_dir, fname)
    mime = ('application/json' if fname.endswith('.json') else
            'text/csv' if fname.endswith('.csv') else
            'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')

    meta = {'name': fname, 'parents': [folder]}
    media = MediaFileUpload(fpath, mimetype=mime, resumable=True)
    f = service.files().create(body=meta, media_body=media, fields='id').execute()
    print(f'Uploaded {fname} → Drive ID {f.get("id")}')
```

#### Step 5: Create the GCP Service Account (One Time)

This service account will be used by GitHub Actions to authenticate with Google Drive and upload files. You need to create this once in your Google Cloud Project.

1.  **Create Service Account:** Open Google Cloud Shell or your local terminal with `gcloud` SDK configured, and run:
    ```bash
    # Replace YOUR_PROJECT_ID with your actual Google Cloud Project ID
    gcloud iam service-accounts create github-pipeline \
        --display-name="GitHub Actions Job Pipeline" \
        --project=YOUR_PROJECT_ID
    ```

2.  **Download JSON Key:** Generate a JSON key for the service account:
    ```bash
    gcloud iam service-accounts keys create sa_key.json \
        --iam-account=github-pipeline@YOUR_PROJECT_ID.iam.gserviceaccount.com
    ```
    This will download a `sa_key.json` file. The *contents* of this file are what you'll paste into the `GDRIVE_SERVICE_ACCOUNT_JSON` GitHub secret.

3.  **Share your Drive outputs folder:**
    *   Open your Google Drive `outputs` folder in your browser.
    *   Click on "Share" (or the person icon).
    *   Paste the service account email: `github-pipeline@YOUR_PROJECT_ID.iam.gserviceaccount.com`
    *   Grant it **Editor** access (important for uploading files).

4.  **Paste JSON into GitHub Secret:** Copy the *entire content* of the `sa_key.json` file you downloaded and paste it as the value for the `GDRIVE_SERVICE_ACCOUNT_JSON` secret in your GitHub repository (as described in Step 2).

Once all these steps are completed, your GitHub Action workflow should be configured to run your notebook weekly and upload the results to your Google Drive!

### Workaround: GitHub Actions with Workload Identity Federation

This section replaces **Step 5: Create the GCP Service Account (One Time)** and involves modifications to **Steps 2, 3, and 4** of the standard GitHub Actions setup, as your organization's policy prevents direct service account key creation. Workload Identity Federation is the recommended and most secure approach in such scenarios.

#### Google Cloud Setup

These steps will be performed in your Google Cloud Project using the `gcloud` CLI (e.g., in Cloud Shell).

##### Step GC1: Verify/Create Service Account

You've already created the `github-pipeline` service account. We'll use this existing service account. If you hadn't created it, the command would be:

```bash
gcloud iam service-accounts create github-pipeline \
    --display-name="GitHub Actions Job Pipeline" \
    --project=project-bf62429c-050a-4ead-b92
```

##### Step GC2: Grant IAM Roles to Service Account

Ensure your `github-pipeline` service account has the necessary permissions to access Google Drive. This involves sharing your Google Drive `outputs` folder with the service account.

1.  **Open your Google Drive `outputs` folder** in your browser.
2.  Click on "Share" (or the person icon).
3.  **Paste the service account email:** `github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com` (replace with your actual project ID if different).
4.  Grant it **Editor** access (important for uploading files).

Additionally, you might want to grant it the `Storage Object Creator` role if you also plan to upload to Google Cloud Storage (though your current setup only uses Drive):

```bash
gcloud projects add-iam-policy-binding project-bf62429c-050a-4ead-b92 \
    --member="serviceAccount:github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com" \
    --role="roles/storage.objectCreator"
```

In [18]:
print('gcloud projects add-iam-policy-binding project-bf62429c-050a-4ead-b92 --member="serviceAccount:github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com" --role="roles/storage.objectCreator"')

gcloud projects add-iam-policy-binding project-bf62429c-050a-4ead-b92 --member="serviceAccount:github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com" --role="roles/storage.objectCreator"


##### Step GC3: Create Workload Identity Pool

A Workload Identity Pool manages external identities that can be granted access to your Google Cloud resources.

```bash
gcloud iam workload-identity-pools create github-pool \
    --project="project-bf62429c-050a-4ead-b92" \
    --location="global" \
    --display-name="GitHub Actions Workload Identity Pool"
```

**Note:** If you get an error that the pool already exists, you can skip this step.

In [19]:
print('gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GitHub Actions Workload Identity Pool"')

gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GitHub Actions Workload Identity Pool"


In [20]:
print('gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions Workload Identity Pool"')

gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions Workload Identity Pool"


In [21]:
print('gcloud iam workload-identity-pools providers create-oidc github-provider --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider" --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository" --issuer-uri="https://token.actions.githubusercontent.com"')

gcloud iam workload-identity-pools providers create-oidc github-provider --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider" --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository" --issuer-uri="https://token.actions.githubusercontent.com"


In [22]:
print('gcloud iam service-accounts add-iam-policy-binding github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com --project="project-bf62429c-050a-4ead-b92" --role="roles/iam.workloadIdentityUser" --member="principal://iam.googleapis.com/projects/$(gcloud projects describe project-bf62429c-050a-4ead-b92 --format=\'value(projectNumber)\')/locations/global/workloadIdentityPools/github-pool/subject/repo:YOUR_GITHUB_ORG/YOUR_GITHUB_REPO:ref:refs/heads/main"')

gcloud iam service-accounts add-iam-policy-binding github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com --project="project-bf62429c-050a-4ead-b92" --role="roles/iam.workloadIdentityUser" --member="principal://iam.googleapis.com/projects/$(gcloud projects describe project-bf62429c-050a-4ead-b92 --format='value(projectNumber)')/locations/global/workloadIdentityPools/github-pool/subject/repo:YOUR_GITHUB_ORG/YOUR_GITHUB_REPO:ref:refs/heads/main"


In [23]:
print('gcloud iam workload-identity-pools list --project="project-bf62429c-050a-4ead-b92" --location="global"')

gcloud iam workload-identity-pools list --project="project-bf62429c-050a-4ead-b92" --location="global"


In [24]:
print('gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions Workload Identity Pool"')

gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions Workload Identity Pool"


After successfully running the above command and confirming the Workload Identity Pool is created, please proceed to re-run **Step GC4** and **Step GC5** from the notebook. Remember to replace `YOUR_GITHUB_ORG` and `YOUR_GITHUB_REPO` with your actual GitHub details in **Step GC5**.

In [25]:
print('gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions WIF Pool"')

gcloud iam workload-identity-pools create github-pool --project="project-bf62429c-050a-4ead-b92" --location="global" --display-name="GH Actions WIF Pool"


##### Step GC4: Create Workload Identity Provider for GitHub

This provider establishes trust with GitHub's OIDC (OpenID Connect) provider.

```bash
gcloud iam workload-identity-pools providers create-oidc github-provider \
    --project="project-bf62429c-050a-4ead-b92" \
    --location="global" \
    --workload-identity-pool="github-pool" \
    --display-name="GitHub Actions OIDC Provider" \
    --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository" \
    --issuer-uri="https://token.actions.githubusercontent.com"
```

**Note:** If you get an error that the provider already exists, you can skip this step.

##### Step GC5: Grant GitHub Identity Access to Impersonate Service Account

This is the critical step where you allow a specific GitHub repository's identity to impersonate your `github-pipeline` service account. Replace `YOUR_GITHUB_ORG` and `YOUR_GITHUB_REPO` with your actual GitHub organization/username and repository name.

```bash
gcloud iam service-accounts add-iam-policy-binding github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com \
    --project="project-bf62429c-050a-4ead-b92" \
    --role="roles/iam.workloadIdentityUser" \
    --member="principal://iam.googleapis.com/projects/$(gcloud projects describe project-bf62429c-050a-4ead-b92 --format='value(projectNumber)')/locations/global/workloadIdentityPools/github-pool/subject/repo:YOUR_GITHUB_ORG/YOUR_GITHUB_REPO:ref:refs/heads/main"
```

**Important:**
*   Replace `YOUR_GITHUB_ORG` with your GitHub username or organization name.
*   Replace `YOUR_GITHUB_REPO` with the name of your GitHub repository.
*   If you're using a different branch than `main`, update `ref:refs/heads/main` accordingly (e.g., `ref:refs/heads/master`).
*   The `$(gcloud projects describe ...)` part automatically fetches your project *number*, which is required here. Ensure you're logged into the correct `gcloud` configuration.

In [26]:
print('gcloud iam workload-identity-pools providers create-oidc github-provider-v2 --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider V2" --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository" --issuer-uri="https://token.actions.githubusercontent.com" --attribute-condition="attribute.repository == \'rpp5069-cpu/job-pipeline\'"')

gcloud iam workload-identity-pools providers create-oidc github-provider-v2 --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider V2" --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository" --issuer-uri="https://token.actions.githubusercontent.com" --attribute-condition="attribute.repository == 'rpp5069-cpu/job-pipeline'"


In [27]:
print('gcloud iam service-accounts add-iam-policy-binding github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com --project="project-bf62429c-050a-4ead-b92" --role="roles/iam.workloadIdentityUser" --member="principal://iam.googleapis.com/projects/$(gcloud projects describe project-bf62429c-050a-4ead-b92 --format=\'value(projectNumber)\')/locations/global/workloadIdentityPools/github-pool/subject/repo:YOUR_GITHUB_ORG/YOUR_GITHUB_REPO:ref:refs/heads/main"')

gcloud iam service-accounts add-iam-policy-binding github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com --project="project-bf62429c-050a-4ead-b92" --role="roles/iam.workloadIdentityUser" --member="principal://iam.googleapis.com/projects/$(gcloud projects describe project-bf62429c-050a-4ead-b92 --format='value(projectNumber)')/locations/global/workloadIdentityPools/github-pool/subject/repo:YOUR_GITHUB_ORG/YOUR_GITHUB_REPO:ref:refs/heads/main"


In [28]:
print('gcloud iam workload-identity-pools providers create-oidc github-provider --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider" --attribute-mapping="google.subject=assertion.sub" --issuer-uri="https://token.actions.githubusercontent.com"')

gcloud iam workload-identity-pools providers create-oidc github-provider --project="project-bf62429c-050a-4ead-b92" --location="global" --workload-identity-pool="github-pool" --display-name="GitHub Actions OIDC Provider" --attribute-mapping="google.subject=assertion.sub" --issuer-uri="https://token.actions.githubusercontent.com"


#### GitHub Actions Setup

Now, let's update your GitHub repository.

##### Step GA1: Update GitHub Repository Secrets

Go to your GitHub repository `Settings` → `Secrets and variables` → `Actions`.

*   **Add a new secret:**
    *   **Name:** `GCP_PROJECT_ID`
    *   **Value:** `project-bf62429c-050a-4ead-b92` (your Google Cloud Project ID)
*   **Remove the `GDRIVE_SERVICE_ACCOUNT_JSON` secret.** It is no longer needed with Workload Identity Federation.

##### Step GA2: Modify `upload_to_drive.py`

Update your `upload_to_drive.py` script to use Google's default credential mechanism, as the environment will now be authenticated via Workload Identity Federation. This means the `GDRIVE_SERVICE_ACCOUNT_JSON` environment variable will no longer be available.

Replace the contents of `upload_to_drive.py` with this updated version:

```python
import os, json
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from google.auth import default # Import default credentials

# Use default credentials provided by Workload Identity Federation
creds, project = default()
service = build('drive', 'v3', credentials=creds)
folder = os.environ['GDRIVE_FOLDER_ID'] # This secret is still needed

output_dir = '/tmp/job_outputs/outputs'
if not os.path.isdir(output_dir): # Corrected: os.path() was incorrect
    output_dir = '/tmp/job_outputs'

for fname in os.listdir(output_dir):
    if not fname.startswith('MASTER_'):
        continue
    fpath = os.path.join(output_dir, fname)
    mime = ('application/json' if fname.endswith('.json') else
            'text/csv' if fname.endswith('.csv') else
            'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')

    meta = {'name': fname, 'parents': [folder]}
    media = MediaFileUpload(fpath, mimetype=mime, resumable=True)
    f = service.files().create(body=meta, media_body=media, fields='id').execute()
    print(f'Uploaded {fname} → Drive ID {f.get("id")}')
```

##### Step GA3: Modify GitHub Actions Workflow YAML (`weekly_job_collection.yml`)

I have updated the `papermill` command to use the specific filename you provided: `CS_MBA_AI_USAJobs_Jooble_Master_Combiner_v3_(2).ipynb`.

```yaml
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: '3.11' }

      # 1. Authenticate via Workload Identity Federation
      - id: 'auth'
        uses: 'google-github-actions/auth@v2'
        with:
          workload_identity_provider: 'projects/71023542519/locations/global/workloadIdentityPools/github-pool/providers/github-provider-v2'
          service_account: 'github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com'

      # 2. Setup gcloud CLI
      - name: 'Set up Cloud SDK'
        uses: 'google-github-actions/setup-gcloud@v2'

      - name: List Repository Files (Debug)
        run: ls -R

      - name: Install dependencies
        run: |
          pip install papermill pandas openpyxl requests beautifulsoup4 tqdm \
            jupyter nbconvert google-auth google-auth-oauthlib \
            google-api-python-client

      - name: Execute notebook
        env:
          USAJOBS_EMAIL: ${{ secrets.USAJOBS_EMAIL }}
          USAJOBS_KEY: ${{ secrets.USAJOBS_KEY }}
          JOOBLE_KEY: ${{ secrets.JOOBLE_KEY }}
        run: |
          mkdir -p job_outputs job_work
          # Using the filename provided by user
          papermill "CS_MBA_AI_USAJobs_Jooble_Master_Combiner_v3_(2).ipynb" \
            ./executed_notebook.ipynb \
            --log-output \
            --no-progress-bar \
            -p DRIVE_BASE job_outputs

      - name: Upload outputs to Google Drive
        env:
          GDRIVE_FOLDER_ID: ${{ secrets.GDRIVE_FOLDER_ID }}
        run: python upload_to_drive.py

      - name: Archive executed notebook
        uses: actions/upload-artifact@v4
        if: always()
        with:
          name: executed-notebook-${{ github.run_id }}
          path: ./executed_notebook.ipynb
```

```javascript
// ── Google Apps Script: Weekly Job Collection ────────────────────────────────
const NOTEBOOK_FILE_ID = 'YOUR_NOTEBOOK_FILE_ID'; // ← paste here
const NOTIFY_EMAIL = Session.getActiveUser().getEmail();

function executeNotebook() {
  const token = ScriptApp.getOAuthToken();
  const url = `https://colab.research.google.com/v2/drives/drive/files/${NOTEBOOK_FILE_ID}:execute`;
  const opts = {
    method: 'post',
    headers: { Authorization: `Bearer ${token}`, 'Content-Type': 'application/json' },
    payload: JSON.stringify({ accelerator_type: 'NONE' }),
    muteHttpExceptions: true,
  };
  const resp = UrlFetchApp.fetch(url, opts);
  const code = resp.getResponseCode();
  Logger.log(`HTTP ${code} — ${resp.getContentText().substring(0, 200)}`);
  return code;
}

function weeklyJobCollection() {
  Logger.log('Run started: ' + new Date());
  const code = executeNotebook();

  const subject = code >= 200 && code < 300
    ? 'Job Pipeline Triggered'
    : `Job Pipeline FAILED (HTTP ${code})`;
  const body = code >= 200 && code < 300
    ? `Pipeline triggered at ${new Date()}. Check Drive/Research/JobPostings/outputs/ in ~35 min.`
    : `Trigger failed (HTTP ${code}). Verify notebook file ID and that notebook is saved in Drive.`;
  GmailApp.sendEmail(NOTIFY_EMAIL, subject, body);
}

// Run this once to create the Monday 6 AM trigger programmatically
function createWeeklyTrigger() {
  ScriptApp.getProjectTriggers()
    .filter(t => t.getHandlerFunction() === 'weeklyJobCollection')
    .forEach(t => ScriptApp.deleteTrigger(t));
  ScriptApp.newTrigger('weeklyJobCollection')
    .timeBased().onWeekDay(ScriptApp.WeekDay.MONDAY).atHour(6).create();
  Logger.log('Trigger created: every Monday at 6 AM.');
}
```

--## Section 9 — Incremental Update (Archive → Append)
**First run:** Sections 4–7 build the initial master.
**All subsequent runs:** uncomment `run_incremental_update()` at the bottom.
Steps per run:
1. Move prior `MASTER_*` files to `archive/`
2. Load prior master hash set as the dedup baseline
3. Filter fresh records (already in memory from Sections 4–5) to new-only
4. Back-fill O\\*NET skills for any pre-v3.1 records
5. Concat, dedup, save updated master

In [30]:
def find_latest_master():
    candidates = sorted(
        [f for f in os.listdir(DRIVE_OUTPUTS) if f.startswith('MASTER_') and
         f.endswith('.json')],
        reverse=True)
    return os.path.join(DRIVE_OUTPUTS, candidates[0]) if candidates else None

def archive_outputs():
    moved = []
    for fname in os.listdir(DRIVE_OUTPUTS):
        if fname.startswith('MASTER_'):
            src = os.path.join(DRIVE_OUTPUTS, fname)
            dst = os.path.join(DRIVE_ARCHIVE, fname)
            shutil.move(src, dst)
            moved.append(fname)
    print(f' Archived: {fname}')
    return moved

def run_incremental_update():
    global df_all, master
    prior_path = find_latest_master()
    if not prior_path:
        print('No prior master — run Sections 4–7 first.')
        return
    print(f'Loading prior master: {os.path.basename(prior_path)}')
    with open(prior_path, encoding='utf-8') as f:
        prior_data = json.load(f)
    prior_postings = prior_data['postings']
    prior_hids \
        = {p['hash_id'] for p in prior_postings}
    print(f' Prior records : {len(prior_postings)}')


    print('Archiving prior files...')
    archive_outputs()
    new_raw = []
    if 'usajobs_records' in dir(): new_raw.extend(usajobs_records)
    if 'jooble_records' in dir(): new_raw.extend(jooble_records)
    if not new_raw:
        print('No fresh records in memory — run Sections 4 and 5 first.')
        return
    new_only = [r for r in new_raw if r['hash_id'] not in prior_hids]
    print(f' Fresh fetched : {len(new_raw)}')
    print(f' New only        : {len(new_only)}')
    # Back-fill O*NET for pre-v3.1 prior records
    backfilled = 0
    for p in prior_postings:
        if not p.get('onet_skills'):
            p['onet_skills'] \
                = map_onet_skills(p.get('title',''),
                                  p.get('description',''), p.get('required_skills',[]))
            p['technical_skills'] = p.get('required_skills', [])
            backfilled += 1
    if backfilled:
        print(f' Back-filled O*NET for {backfilled} prior records')
    if not new_only:
        print('No new records — master is current.')
        return
    FIELDS = ['hash_id','job_title','company','location','required_skills',
              'technical_skills','onet_skills','domain','era','publication_date',
              'application_close','salary','job_type','grade','job_url',
              'description','relevance_score','matched_terms','relevance_tier',
              'source','search_keyword']
    prior_rows = []
    for p in prior_postings:
        row = {f: p.get(f if f != 'publication_date' else 'posted_date', p.get(f, ''))
               for f in FIELDS}
        row['job_title'] = p.get('title', p.get('job_title', ''))
        prior_rows.append(row)
    df_all = pd.concat([pd.DataFrame(prior_rows), pd.DataFrame(new_only)],
                       ignore_index=True)
    df_all = (df_all.sort_values('relevance_score', ascending=False)
              .drop_duplicates(subset='hash_id', keep='first')
              .reset_index(drop=True))
    df_all.insert(0, 'id', range(1, len(df_all) + 1))
    net_new = len(df_all) - len(prior_postings)
    print(f' Updated total : {len(df_all)} records (+{net_new} net new)')


    master \
        = build_master_json(df_all)
    loc_json = os.path.join(LOCAL_WORK, MASTER_JSON)
    with open(loc_json, 'w', encoding='utf-8') as f:
        json.dump(master, f, indent=2, ensure_ascii=False)
    save_to_drive(loc_json, DRIVE_OUTPUTS, MASTER_JSON)
    write_run_log(df_all)
    print(f'Incremental update complete. Re-run Section 7 to regenerate CSV + XLSX.')

print("Incremental update ready.")
print(" First run : Sections 4–7 build the initial master.")
print(" Subsequent : uncomment run_incremental_update() below.")
# ── Uncomment for all runs AFTER the first master is built ────────────────────
# run_incremental_update()

Incremental update ready.
 First run : Sections 4–7 build the initial master.
 Subsequent : uncomment run_incremental_update() below.


In [ ]:
print('gcloud iam service-accounts create github-pipeline --display-name="GitHub Actions Job Pipeline" --project=project-bf62429c-050a-4ead-b92')

---
## Section 10A — Phase 1: Dashboard Bootstrap
**Goal:** Install the interactive rendering stack, define the shared theme config, and create the `FilterState` dataclass that all nine dashboard phases read from. Run this cell once before launching any Phase 2–9 cells.

| Package | Role |
|---|---|
| `plotly` | Interactive, animated charts (replaces static matplotlib) |
| `ipywidgets` | Core widget primitives (sliders, buttons, HTML output) |
| `ipyvuetify` | Material Design components (cards, dialogs, tabs, snackbars) |
| `kaleido` | Plotly static image export for PDF reports |
| `reportlab` | PDF report generation (Phase 8) |
| `voila` | Converts the notebook to a standalone web app (Phase 9) |


In [31]:
# Phase 1 — Install dashboard stack
!pip install -q plotly ipywidgets ipyvuetify kaleido reportlab voila
print('Dashboard stack ready.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 92.7 MB/s eta 0:00:00
Dashboard stack ready.


In [32]:
# Phase 1 — Shared theme, imports, and FilterState
import threading, base64, io, os, json, warnings
from dataclasses import dataclass, field
from collections import Counter
from itertools import chain
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
warnings.filterwarnings('ignore')

# ── Global theme ─────────────────────────────────────────────────────────────
THEME = {
    'primary'   : '#1F3864',
    'accent'    : '#2A78D6',
    'success'   : '#1BAF7A',
    'warn'      : '#EDA100',
    'danger'    : '#E34948',
    'purple'    : '#4A3AA7',
    'bg'        : '#F8F9FB',
    'surface'   : '#FFFFFF',
    'muted'     : '#898781',
    'border'    : '#E1E0D9',
    'font'      : 'Arial, sans-serif',
    'anim_ms'   : 400,
    # Plotly series palette (Tidepool)
    'series'    : ['#2a78d6','#1baf7a','#eda100','#4a3aa7',
                   '#e34948','#eb6834','#e87ba4','#008300'],
}

# ── Filter state dataclass ────────────────────────────────────────────────────
@dataclass
class FilterState:
    domain    : str   = 'ALL'
    tier      : list  = field(default_factory=lambda: ['HIGH','MEDIUM','LOW'])
    source    : str   = 'ALL'
    era       : str   = 'ALL'
    score_min : int   = 0
    score_max : int   = 12
    keyword   : str   = ''

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a filtered copy of df according to current state."""
        out = df.copy()
        if self.domain != 'ALL':
            out = out[out['domain'] == self.domain]
        if self.tier:
            out = out[out['relevance_tier'].isin(self.tier)]
        if self.source != 'ALL':
            out = out[out['source'] == self.source]
        if self.era != 'ALL':
            out = out[out['era'] == self.era]
        out = out[
            (out['relevance_score'] >= self.score_min) &
            (out['relevance_score'] <= self.score_max)
        ]
        if self.keyword.strip():
            kw = self.keyword.strip().lower()
            mask = (
                out['job_title'].str.lower().str.contains(kw, na=False) |
                out['description'].str.lower().str.contains(kw, na=False)
            )
            out = out[mask]
        return out

state = FilterState()

# ── Guard: ensure df_all exists (needed if running dashboard before data fetch) ──
if 'df_all' not in dir() or df_all is None or len(df_all) == 0:
    print('⚠  df_all not found — generating synthetic sample for dashboard preview.')
    import numpy as np
    rng = np.random.default_rng(42)
    N = 500
    domains  = rng.choice(['CS','MBA','DATA_ANALYTICS','AI'], N, p=[.28,.18,.26,.28])
    tiers    = rng.choice(['HIGH','MEDIUM','LOW'], N, p=[.38,.35,.27])
    sources  = rng.choice(['USAJobs','Jooble'], N, p=[.43,.57])
    eras     = rng.choice(['current','historical'], N, p=[.65,.35])
    scores   = rng.integers(2,13,N)
    titles   = rng.choice([
        'Software Engineer','Data Scientist','ML Engineer','Product Manager',
        'Financial Analyst','Cloud Engineer','NLP Engineer','Business Analyst',
        'DevOps Engineer','Data Analyst','AI Researcher','Program Analyst',
    ], N)
    companies = rng.choice(['Google','DoD','NSA','Deloitte','Amazon','JPMorgan','NASA','GSA'], N)
    ONET_SAMPLE = ['Programming','Critical Thinking','Technology Design',
                   'Mathematics','Coordination','Active Learning',
                   'Complex Problem Solving','Writing','Science','Monitoring']
    def rand_onet():
        k = int(rng.integers(2,7))
        return list(rng.choice(ONET_SAMPLE, k, replace=False))
    def rand_skills():
        pool = ['Python','SQL','AWS','Docker','TensorFlow','Tableau','Agile']
        k = int(rng.integers(1,5))
        return list(rng.choice(pool, k, replace=False))
    df_all = pd.DataFrame({
        'id'              : range(1, N+1),
        'job_title'       : titles,
        'company'         : companies,
        'location'        : rng.choice(['Washington, DC, US','Remote, US','Arlington, VA, US','San Francisco, CA, US'], N),
        'domain'          : domains,
        'relevance_tier'  : tiers,
        'relevance_score' : scores,
        'source'          : sources,
        'era'             : eras,
        'publication_date': pd.date_range('2025-06-01', periods=N, freq='3h').strftime('%Y-%m-%d'),
        'onet_skills'     : [rand_onet() for _ in range(N)],
        'required_skills' : [rand_skills() for _ in range(N)],
        'description'     : ['Sample job description text.' for _ in range(N)],
        'salary'          : rng.choice(['$80,000-$120,000 PA','$100,000-$150,000 PA',''], N),
        'job_url'         : ['' for _ in range(N)],
        'matched_terms'   : ['python|sql' for _ in range(N)],
    })
    print(f'  Synthetic df_all: {len(df_all)} rows')

# Ensure onet_skills is always a list column
def _to_list(x):
    if isinstance(x, list): return x
    if isinstance(x, str):
        try: return json.loads(x)
        except: return [s.strip() for s in x.strip('[]').replace("'",'').split(',') if s.strip()]
    return []

df_all['onet_skills']     = df_all['onet_skills'].apply(_to_list)
df_all['required_skills'] = df_all['required_skills'].apply(_to_list)

print(f'Phase 1 complete. df_all: {len(df_all):,} rows | FilterState ready.')


Phase 1 complete. df_all: 6,881 rows | FilterState ready.


---
## Section 10B — Phase 2: Animated KPI Cards
Four primary metric cards count up from 0 to their target value when rendered and re-animate whenever the domain filter changes. The `animate_counter` function uses a `threading.Timer` loop at ~60 fps.


In [33]:
# Phase 2 — Animated KPI cards

_P2_CARDS = {}  # widget references keyed by metric name
_P2_TIMERS = {} # debounce timers

def _kpi_metrics(df):
    """Compute the four headline KPI values from a filtered DataFrame."""
    total   = len(df)
    current = int((df['era'] == 'current').sum()) if total else 0
    high    = int((df['relevance_tier'] == 'HIGH').sum()) if total else 0
    avg_s   = round(df['relevance_score'].mean(), 1) if total else 0.0
    return {
        'Total postings'      : (total,    f'{high/max(total,1)*100:.0f}% HIGH relevance', THEME['accent']),
        'Current (≥ 2026)'    : (current,  f'{current/max(total,1)*100:.0f}% of total',    THEME['success']),
        'HIGH tier postings'  : (high,     f'{total - high} not HIGH tier',                THEME['purple']),
        'Avg relevance score' : (avg_s,    f'out of 12  |  min {df["relevance_score"].min() if total else 0}',
                                            THEME['warn']),
    }

def animate_counter(out_widget, target, fmt_fn=None, duration_ms=None):
    """Smoothly count up from 0 → target in out_widget at ~60 fps."""
    duration_ms = duration_ms or THEME['anim_ms']
    fmt  = fmt_fn or (lambda v: f'{int(v):,}')
    steps = max(1, int(duration_ms / 16))
    delta = target / max(steps, 1)
    def tick(step=0):
        if step > steps:
            out_widget.value = f'<span style="font-size:24px;font-weight:500;color:{THEME["primary"]}">{fmt(target)}</span>'
            return
        val = delta * step
        out_widget.value = f'<span style="font-size:24px;font-weight:500;color:{THEME["primary"]}">{fmt(val)}</span>'
        threading.Timer(0.016, tick, [step + 1]).start()
    tick()

def _make_kpi_card(label, value, subtitle, color):
    """Return an HBox card widget with animated counter."""
    accent_bar = widgets.HTML(
        value=f'<div style="width:4px;min-height:72px;background:{color};border-radius:4px 0 0 4px;"></div>'
    )
    counter_html = widgets.HTML(value='<span style="font-size:24px;font-weight:500;">0</span>')
    label_html   = widgets.HTML(value=f'<div style="font-size:11px;color:{THEME["muted"]};margin-bottom:2px">{label}</div>')
    sub_html     = widgets.HTML(value=f'<div style="font-size:11px;color:{THEME["muted"]};margin-top:4px">{subtitle}</div>')
    inner = widgets.VBox([label_html, counter_html, sub_html],
        layout=widgets.Layout(padding='10px 14px', flex='1'))
    card  = widgets.HBox([accent_bar, inner],
        layout=widgets.Layout(
            border=f'1px solid {THEME["border"]}',
            border_radius='10px',
            background_color=THEME['surface'],
            min_width='160px',
            flex='1',
        ))
    fmt_fn = (lambda v: f'{v:.1f}') if isinstance(value, float) else None
    return card, counter_html, value, fmt_fn

def render_kpi_row(df):
    """Build and display the KPI row; store references for later updates."""
    metrics = _kpi_metrics(df)
    cards = []
    for label, (val, sub, col) in metrics.items():
        card, counter_widget, target, fmt_fn = _make_kpi_card(label, val, sub, col)
        _P2_CARDS[label] = (counter_widget, target, fmt_fn)
        cards.append(card)
    row = widgets.HBox(cards, layout=widgets.Layout(
        display='flex', flex_flow='row wrap', gap='10px', width='100%'
    ))
    display(row)
    for label, (counter_widget, target, fmt_fn) in _P2_CARDS.items():
        animate_counter(counter_widget, target, fmt_fn)

def update_kpi_row(df):
    """Re-animate existing KPI cards with new values (no DOM rebuild)."""
    metrics = _kpi_metrics(df)
    for label, (val, sub, col) in metrics.items():
        if label in _P2_CARDS:
            counter_widget, _, fmt_fn = _P2_CARDS[label]
            animate_counter(counter_widget, val, fmt_fn)

_P2_OUT = widgets.Output()
with _P2_OUT:
    render_kpi_row(state.apply(df_all))
display(_P2_OUT)
print('Phase 2 ready — KPI cards animated.')


Output()

Phase 2 ready — KPI cards animated.


---
## Section 10C — Phase 3: Interactive Plotly Charts
Eight `FigureWidget` objects replace the static matplotlib outputs. All charts share an `update_all_charts(df)` function so a single filter change re-animates every chart simultaneously using `transition={'duration': 400, 'easing': 'cubic-in-out'}`.


In [35]:
# Phase 3 — Build all eight Plotly FigureWidgets
import plotly.graph_objects as go

_LAYOUT_BASE = dict(
    paper_bgcolor='rgba(0,0,0,0)',
    plot_bgcolor ='rgba(0,0,0,0)',
    font=dict(family=THEME['font'], color=THEME['primary'], size=11),
    margin=dict(l=8, r=8, t=36, b=8)
)
_AXIS = dict(gridcolor=THEME['border'], linecolor=THEME['border'],
             tickfont=dict(size=10), zeroline=False)
_TRANS = dict(duration=THEME['anim_ms'], easing='cubic-in-out')

DOMAINS_ORDER = ['CS', 'MBA', 'DATA_ANALYTICS', 'AI']

# ── Chart 1: Domain bar ───────────────────────────────────────────────────────
def _dom_data(df):
    counts = df['domain'].value_counts().reindex(DOMAINS_ORDER, fill_value=0)
    return counts.index.tolist(), counts.values.tolist()

_dom_lbls, _dom_vals = _dom_data(df_all)
fig_dom = go.FigureWidget(data=[
    go.Bar(x=_dom_lbls, y=_dom_vals,
           marker_color=THEME['series'][:4],
           marker_line_width=0,
           text=_dom_vals, textposition='outside', textfont_size=10)
])
fig_dom.update_layout(**_LAYOUT_BASE,
    title=dict(text='Postings by domain', font_size=13),
    xaxis=dict(**_AXIS), yaxis=dict(**_AXIS, title=''),
    height=220)

# ── Chart 2: Relevance tier horizontal bar ────────────────────────────────────
def _tier_data(df):
    counts = df['relevance_tier'].value_counts().reindex(['HIGH','MEDIUM','LOW'], fill_value=0)
    return counts.index.tolist(), counts.values.tolist()

_t_lbls, _t_vals = _tier_data(df_all)
fig_tier = go.FigureWidget(data=[
    go.Bar(y=_t_lbls, x=_t_vals, orientation='h',
           marker_color=[THEME['accent'], THEME['success'], THEME['warn']],
           text=_t_vals, textposition='outside', textfont_size=10)
])
fig_tier.update_layout(**_LAYOUT_BASE,
    title=dict(text='Relevance tiers', font_size=13),
    xaxis=dict(**_AXIS), yaxis=dict(**_AXIS), height=200)

# ── Chart 3: Source doughnut ──────────────────────────────────────────────────
def _src_data(df):
    vc = df['source'].value_counts()
    labels = vc.index.tolist()
    vals   = vc.values.tolist()
    return labels, vals

_s_lbls, _s_vals = _src_data(df_all)
fig_src = go.FigureWidget(data=[
    go.Pie(labels=_s_lbls, values=_s_vals,
           hole=0.55,
           marker_colors=[THEME['accent'], THEME['success']],
           textfont_size=11)
])
fig_src.update_layout(**_LAYOUT_BASE,
    title=dict(text='Source split', font_size=13),
    showlegend=True,
    legend=dict(orientation='h', y=-0.12, font_size=10),
    height=230)

# ── Chart 4: Era doughnut ─────────────────────────────────────────────────────
def _era_data(df):
    vc = df['era'].value_counts()
    return vc.index.tolist(), vc.values.tolist()

_e_lbls, _e_vals = _era_data(df_all)
fig_era = go.FigureWidget(data=[
    go.Pie(labels=_e_lbls, values=_e_vals,
           hole=0.55,
           marker_colors=[THEME['accent'], THEME['muted']],
           textfont_size=11)
])
fig_era.update_layout(**_LAYOUT_BASE,
    title=dict(text='Posting era', font_size=13),
    showlegend=True,
    legend=dict(orientation='h', y=-0.12, font_size=10),
    height=230)

# ── Chart 5: Score distribution line ─────────────────────────────────────────
def _score_data(df):
    vc = df['relevance_score'].value_counts().sort_index()
    return vc.index.tolist(), vc.values.tolist()

_sc_x, _sc_y = _score_data(df_all)
fig_score = go.FigureWidget(data=[
    go.Scatter(x=_sc_x, y=_sc_y, mode='lines+markers',
               line=dict(color=THEME['accent'], width=2),
               fill='tozeroy',
               fillcolor=f'rgba(42,120,214,0.10)',
               marker=dict(size=6, color=THEME['accent']))
])
fig_score.update_layout(**_LAYOUT_BASE,
    title=dict(text='Score distribution', font_size=13),
    xaxis=dict(**_AXIS, title='Relevance score'),
    yaxis=dict(**_AXIS, title='# postings'),
    height=220)

# ── Chart 6: O*NET frequency bar ──────────────────────────────────────────────
def _onet_data(df, top_n=12):
    flat = list(chain.from_iterable(df['onet_skills'].dropna()))
    mc = Counter(flat).most_common(top_n)
    if not mc: return [], []
    labels = [r[0] for r in mc][::-1]
    vals   = [r[1] for r in mc][::-1]
    return labels, vals

_o_lbls, _o_vals = _onet_data(df_all)
_o_colors = [THEME['series'][i % len(THEME['series'])] for i in range(len(_o_lbls))]
fig_onet = go.FigureWidget(data=[
    go.Bar(y=_o_lbls, x=_o_vals, orientation='h',
           marker_color=_o_colors,
           text=_o_vals, textposition='outside', textfont_size=9)
])
fig_onet.update_layout(**_LAYOUT_BASE,
    title=dict(text='Top O*NET 24.2 competencies', font_size=13),
    xaxis=dict(**_AXIS, title='# postings'),
    yaxis=dict(**_AXIS),
    height=380)

# ── Chart 7: Weekly trend line ────────────────────────────────────────────────
def _trend_data(df):
    if 'publication_date' not in df.columns: return [], []
    df2 = df.copy()
    df2['week'] = pd.to_datetime(df2['publication_date'], errors='coerce').dt.to_period('W').astype(str)
    vc = df2['week'].value_counts().sort_index().tail(12)
    return vc.index.tolist(), vc.values.tolist()

_w_x, _w_y = _trend_data(df_all)
fig_trend = go.FigureWidget(data=[
    go.Scatter(x=_w_x, y=_w_y, mode='lines+markers',
               line=dict(color=THEME['success'], width=2),
               fill='tozeroy',
               fillcolor='rgba(27,175,122,0.10)',
               marker=dict(size=7, color=THEME['success']))
])
fig_trend.update_layout(**_LAYOUT_BASE,
    title=dict(text='Weekly ingestion trend', font_size=13),
    xaxis=dict(**_AXIS, title='Week', tickangle=-30, tickfont_size=9),
    yaxis=dict(**_AXIS, title='# postings'),
    height=220)

# ── Chart 8: Top technical skills bar ────────────────────────────────────────
def _skill_data(df, top_n=12):
    flat = list(chain.from_iterable(df['required_skills'].dropna()))
    mc = Counter(flat).most_common(top_n)
    if not mc: return [], []
    labels = [r[0] for r in mc][::-1]
    vals   = [r[1] for r in mc][::-1]
    return labels, vals

_sk_lbls, _sk_vals = _skill_data(df_all)
fig_skills = go.FigureWidget(data=[
    go.Bar(y=_sk_lbls, x=_sk_vals, orientation='h',
           marker_color=THEME['purple'],
           text=_sk_vals, textposition='outside', textfont_size=9)
])
fig_skills.update_layout(**_LAYOUT_BASE,
    title=dict(text='Top technical skills', font_size=13),
    xaxis=dict(**_AXIS, title='# postings'),
    yaxis=dict(**_AXIS),
    height=380)

# ── Master chart updater (called on every filter change) ─────────────────────
def update_all_charts(df):
    """Re-render all 8 FigureWidgets from a filtered DataFrame with animated transitions."""
    # Chart 1 — domain
    lbls, vals = _dom_data(df)
    with fig_dom.batch_update():
        fig_dom.data[0].x = lbls
        fig_dom.data[0].y = vals
        fig_dom.data[0].text = vals
    # Chart 2 — tiers
    t_lbls, t_vals = _tier_data(df)
    with fig_tier.batch_update():
        fig_tier.data[0].y = t_lbls
        fig_tier.data[0].x = t_vals
        fig_tier.data[0].text = t_vals
    # Chart 3 — source
    s_lbls, s_vals = _src_data(df)
    with fig_src.batch_update():
        fig_src.data[0].labels = s_lbls
        fig_src.data[0].values = s_vals
    # Chart 4 — era
    e_lbls, e_vals = _era_data(df)
    with fig_era.batch_update():
        fig_era.data[0].labels = e_lbls
        fig_era.data[0].values = e_vals
    # Chart 5 — score distribution
    sc_x, sc_y = _score_data(df)
    with fig_score.batch_update():
        fig_score.data[0].x = sc_x
        fig_score.data[0].y = sc_y
    # Chart 6 — O*NET
    o_lbls, o_vals = _onet_data(df)
    with fig_onet.batch_update():
        fig_onet.data[0].y = o_lbls
        fig_onet.data[0].x = o_vals
        fig_onet.data[0].text = o_vals
        fig_onet.data[0].marker.color = [THEME['series'][i % len(THEME['series'])] for i in range(len(o_lbls))]
    # Chart 7 — trend
    w_x, w_y = _trend_data(df)
    with fig_trend.batch_update():
        fig_trend.data[0].x = w_x
        fig_trend.data[0].y = w_y
    # Chart 8 — skills
    sk_lbls, sk_vals = _skill_data(df)
    with fig_skills.batch_update():
        fig_skills.data[0].y = sk_lbls
        fig_skills.data[0].x = sk_vals
        fig_skills.data[0].text = sk_vals

print('Phase 3 ready — 8 Plotly FigureWidgets initialised.')


Phase 3 ready — 8 Plotly FigureWidgets initialised.


---
## Section 10D — Phase 4: Interactive Filter Controls
Domain toggle buttons, tier checkboxes, source dropdown, era toggle, relevance score range slider, and debounced keyword search. Every change propagates through `FilterState.apply()` and calls `update_all_charts()` + `update_kpi_row()`.


In [37]:
# Phase 4 — Filter control panel

# ── Domain toggle buttons ─────────────────────────────────────────────────────
_dom_opts = ['ALL', 'CS', 'MBA', 'DATA_ANALYTICS', 'AI']
_dom_btns = [
    widgets.Button(
        description=d,
        layout=widgets.Layout(height='36px', min_width='90px'),
        style={'button_color': THEME['accent'] if d == 'ALL' else '#FFFFFF',
               'font_color': '#FFFFFF' if d == 'ALL' else THEME['primary']}
    ) for d in _dom_opts
]

def _on_domain(btn):
    state.domain = btn.description
    for b in _dom_btns:
        is_active = b.description == state.domain
        b.style.button_color = THEME['accent'] if is_active else '#FFFFFF'
        b.style.font_color   = '#FFFFFF' if is_active else THEME['primary']
    _apply_filters()

for b in _dom_btns:
    b.on_click(_on_domain)

dom_row = widgets.HBox(_dom_btns,
    layout=widgets.Layout(flex_flow='row wrap', gap='6px', margin='0 0 8px 0'))

# ── Tier checkboxes ───────────────────────────────────────────────────────────
_tier_checks = [
    widgets.Checkbox(value=True, description=t, indent=False,
                     layout=widgets.Layout(width='90px', margin='0'))
    for t in ['HIGH', 'MEDIUM', 'LOW']
]
tier_row = widgets.HBox(
    [widgets.Label('Tier:', layout=widgets.Layout(width='40px'))] + _tier_checks,
    layout=widgets.Layout(margin='0 0 6px 0')
)

def _on_tier(change):
    state.tier = [c.description for c in _tier_checks if c.value]
    _apply_filters()

for c in _tier_checks:
    c.observe(_on_tier, names='value')

# ── Source dropdown ───────────────────────────────────────────────────────────
src_dd = widgets.Dropdown(
    options=['ALL', 'USAJobs', 'Jooble'],
    value='ALL',
    description='Source:',
    style={'description_width': '52px'},
    layout=widgets.Layout(width='200px', margin='0 0 6px 0')
)
src_dd.observe(lambda c: [setattr(state, 'source', c['new']), _apply_filters()], names='value')

# ── Era toggle ────────────────────────────────────────────────────────────────
era_dd = widgets.Dropdown(
    options=['ALL', 'current', 'historical'],
    value='ALL',
    description='Era:',
    style={'description_width': '36px'},
    layout=widgets.Layout(width='180px', margin='0 0 6px 0')
)
era_dd.observe(lambda c: [setattr(state, 'era', c['new']), _apply_filters()], names='value')

# ── Score range slider ────────────────────────────────────────────────────────
score_slider = widgets.IntRangeSlider(
    value=[0, 12], min=0, max=12, step=1,
    description='Score:',
    style={'description_width': '46px'},
    continuous_update=False,
    layout=widgets.Layout(width='320px', margin='0 0 6px 0')
)

def _on_score(change):
    state.score_min, state.score_max = change['new']
    _apply_filters()

score_slider.observe(_on_score, names='value')

# ── Debounced keyword search ───────────────────────────────────────────────────
kw_text = widgets.Text(
    placeholder='Search titles & descriptions…',
    description='',
    layout=widgets.Layout(width='280px', margin='0 0 6px 0')
)
_kw_timer = None

def _on_keyword(change):
    global _kw_timer
    if _kw_timer:
        _kw_timer.cancel()
    state.keyword = change['new']
    _kw_timer = threading.Timer(0.35, _apply_filters)
    _kw_timer.start()

kw_text.observe(_on_keyword, names='value')

# ── Result count badge ────────────────────────────────────────────────────────
_count_html = widgets.HTML(value='...')

def _update_count_badge(df):
    total = len(df_all)
    shown = len(df)
    pct   = shown / max(total, 1)
    color = THEME['success'] if pct > 0.5 else (THEME['warn'] if pct > 0.2 else THEME['danger'])
    _count_html.value = (
        f'<span style="font-size:12px;color:{color};font-weight:500">'
        f'{shown:,} of {total:,} postings shown'
        f'</span>'
    )

# ── Reset button ──────────────────────────────────────────────────────────────
reset_btn = widgets.Button(
    description='↺  Reset filters',
    layout=widgets.Layout(height='36px', width='140px'),
    style={'button_color': '#FFFFFF', 'font_color': THEME['danger']}
)

def _on_reset(btn):
    state.domain    = 'ALL'
    state.tier      = ['HIGH','MEDIUM','LOW']
    state.source    = 'ALL'
    state.era       = 'ALL'
    state.score_min = 0
    state.score_max = 12
    state.keyword   = ''
    # Sync widget values
    for b in _dom_btns:
        is_active = b.description == 'ALL'
        b.style.button_color = THEME['accent'] if is_active else '#FFFFFF'
        b.style.font_color   = '#FFFFFF' if is_active else THEME['primary']
    for c in _tier_checks:
        c.value = True
    src_dd.value    = 'ALL'
    era_dd.value    = 'ALL'
    score_slider.value = (0, 12)
    kw_text.value   = ''
    _apply_filters()

reset_btn.on_click(_on_reset)

# ── Master apply function ─────────────────────────────────────────────────────
def _apply_filters():
    df_f = state.apply(df_all)
    update_kpi_row(df_f)
    update_all_charts(df_f)
    _update_count_badge(df_f)
    # _update_all_insights(df_f) # Commented out to prevent NameError, will be called in Section 10J

# ── Assemble filter panel ─────────────────────────────────────────────────────
_filter_label = widgets.HTML(
    value=f'<div style="font-size:13px;font-weight:500;color:{THEME["primary"]};'
           'margin-bottom:8px">🎛  Filters</div>'
)
filter_panel = widgets.VBox([
    _filter_label,
    widgets.HTML(value='<div style="font-size:11px;color:#898781;margin-bottom:4px">Domain</div>'),
    dom_row,
    tier_row,
    widgets.HBox([src_dd, widgets.HTML(value='&nbsp;'), era_dd]),
    score_slider,
    kw_text,
    widgets.HBox([reset_btn, widgets.HTML(value='&nbsp;'), _count_html],
                 layout=widgets.Layout(align_items='center')),
],
    layout=widgets.Layout(
        border=f'1px solid {THEME["border"]}',
        border_radius='12px',
        padding='14px',
        background_color=THEME['surface'],
        margin='0 0 16px 0',
        width='100%',
    )
)
display(filter_panel)
_apply_filters()   # initial sync
print('Phase 4 ready — filter panel active.')


Phase 4 ready — filter panel active.


---
## Section 10E — Phase 5: O\*NET Competency Spotlight Panel
Clicking any bar in the O\*NET frequency chart opens a detail dialog showing: competency summary, aggregate stats for postings with that skill, cross-domain distribution, and a sample of matching job titles.


In [38]:
# Phase 5 — O*NET spotlight click handler

_spotlight_out = widgets.Output()

def show_onet_spotlight(skill_name: str, df_filtered: pd.DataFrame = None):
    """Render a detail panel for a clicked O*NET skill."""
    df_f  = df_filtered if df_filtered is not None else state.apply(df_all)
    mask  = df_f['onet_skills'].apply(lambda lst: skill_name in lst if isinstance(lst, list) else False)
    df_sk = df_f[mask]
    count = len(df_sk)
    pct   = count / max(len(df_f), 1) * 100
    avg_s = df_sk['relevance_score'].mean() if count else 0
    top_domain = df_sk['domain'].value_counts().idxmax() if count else 'N/A'

    # Cross-domain breakdown
    dom_counts = df_sk['domain'].value_counts().reindex(DOMAINS_ORDER, fill_value=0)
    dom_fig = go.FigureWidget(data=[
        go.Bar(
            x=dom_counts.index.tolist(),
            y=dom_counts.values.tolist(),
            marker_color=THEME['series'][:4],
            text=dom_counts.values.tolist(),
            textposition='outside', textfont_size=10,
        )
    ])
    dom_fig.update_layout(**_LAYOUT_BASE,
        title=dict(text='Cross-domain distribution', font_size=12),
        height=180, margin=dict(l=4, r=4, t=28, b=4),
        xaxis=dict(**_AXIS), yaxis=dict(**_AXIS))

    # Sample titles
    sample_titles = df_sk['job_title'].dropna().unique()[:20].tolist()
    pills_html = ' '.join(
        f'<span style="display:inline-block;padding:3px 10px;margin:3px;'
        f'border:1px solid {THEME["border"]};border-radius:14px;'
        f'font-size:11px;color:{THEME["primary"]};background:{THEME["bg"]}">{t}</span>'
        for t in sample_titles
    )

    header = widgets.HTML(value=(
        f'<div style="border-left:4px solid {THEME["accent"]};padding-left:12px;margin-bottom:12px">'
        f'<div style="font-size:17px;font-weight:500;color:{THEME["primary"]}">{skill_name}</div>'
        f'<div style="font-size:12px;color:{THEME["muted"]}">O*NET 24.2 Competency</div>'
        f'</div>'
    ))
    stats = widgets.HTML(value=(
        f'<div style="display:flex;gap:20px;flex-wrap:wrap;margin-bottom:12px">'
        f'<div><div style="font-size:10px;color:{THEME["muted"]}">Postings with skill</div>'
        f'<div style="font-size:20px;font-weight:500;color:{THEME["accent"]}">{count:,}</div></div>'
        f'<div><div style="font-size:10px;color:{THEME["muted"]}">% of filtered total</div>'
        f'<div style="font-size:20px;font-weight:500;color:{THEME["success"]}">{pct:.1f}%</div></div>'
        f'<div><div style="font-size:10px;color:{THEME["muted"]}">Avg relevance score</div>'
        f'<div style="font-size:20px;font-weight:500;color:{THEME["warn"]}">{avg_s:.1f}</div></div>'
        f'<div><div style="font-size:10px;color:{THEME["muted"]}">Top domain</div>'
        f'<div style="font-size:20px;font-weight:500;color:{THEME["purple"]}">{top_domain}</div></div>'
        f'</div>'
    ))
    titles_section = widgets.HTML(value=(
        f'<div style="margin-top:12px">'
        f'<div style="font-size:12px;color:{THEME["muted"]};margin-bottom:6px">Sample matching job titles</div>'
        f'<div>{pills_html}</div>'
        f'</div>'
    ))
    close_btn = widgets.Button(
        description='✕  Close',
        layout=widgets.Layout(width='100px', height='32px', margin='12px 0 0 0'),
        style={'button_color': '#FFFFFF', 'font_color': THEME['muted']}
    )
    panel = widgets.VBox(
        [header, stats, dom_fig, titles_section, close_btn],
        layout=widgets.Layout(
            border=f'2px solid {THEME["accent"]}',
            border_radius='12px',
            padding='18px',
            background_color=THEME['surface'],
            margin='12px 0',
        )
    )
    close_btn.on_click(lambda _: _spotlight_out.clear_output())
    with _spotlight_out:
        clear_output(wait=True)
        display(panel)

# ── Wire click event to the O*NET FigureWidget ────────────────────────────────
def _on_onet_click(trace, points, selector):
    if not points.point_inds:
        return
    idx    = points.point_inds[0]
    skill  = trace.y[idx] if hasattr(trace, 'y') and trace.y else None
    if skill:
        show_onet_spotlight(skill, state.apply(df_all))

fig_onet.data[0].on_click(_on_onet_click)

display(widgets.HTML(
    value=f'<div style="font-size:12px;color:{THEME["muted"]};margin:6px 0">'
           '💡 Click any bar in the O*NET chart to open the competency spotlight panel.</div>'
))
display(_spotlight_out)
print('Phase 5 ready — O*NET click handler active.')


HTML(value='<div style="font-size:12px;color:#898781;margin:6px 0">💡 Click any bar in the O*NET chart to open …

Output()

Phase 5 ready — O*NET click handler active.


---
## Section 10F — Phase 6: Narrative Annotations & Guided Tour
Each chart receives a computed one-sentence insight string beneath it. A five-step guided tour walks through the data story in sequence.


In [39]:
# Phase 6 — Narrative annotations (auto-computed from filtered data)

_insight_widgets = {}

def _style_insight(text, color=None):
    col = color or THEME['muted']
    return (
        f'<div style="font-size:11.5px;color:{col};font-style:italic;'
        f'padding:6px 10px;border-left:3px solid {THEME["border"]};margin-top:4px">'
        f'{text}</div>'
    )

def insight_domain(df):
    if len(df) == 0: return 'No data for current filters.'
    vc = df['domain'].value_counts()
    top = vc.idxmax(); pct = int(vc.max() / len(df) * 100)
    return f'{top} represents {pct}% of postings — the largest single-domain share under current filters.'

def insight_tiers(df):
    if len(df) == 0: return 'No data for current filters.'
    high_pct = int((df['relevance_tier']=='HIGH').sum() / len(df) * 100)
    med_n    = int((df['relevance_tier']=='MEDIUM').sum())
    return f'{high_pct}% of postings score HIGH relevance; {med_n:,} additional postings qualify as MEDIUM.'

def insight_source(df):
    if len(df) == 0: return 'No data for current filters.'
    vc = df['source'].value_counts()
    if len(vc) < 2: return f'All {len(df):,} postings come from {vc.index[0]}.'
    a, b = vc.index[0], vc.index[1]
    ratio = round(vc[a] / max(vc[b], 1), 1)
    return f'{a} contributes {ratio}× more postings than {b} under current filters.'

def insight_era(df):
    if len(df) == 0: return 'No data for current filters.'
    cur_pct = int((df['era']=='current').sum() / len(df) * 100)
    return f'{cur_pct}% of postings are current (published ≥ Jan 2026), reflecting active labor market demand.'

def insight_score(df):
    if len(df) == 0: return 'No data for current filters.'
    med = df['relevance_score'].median()
    mean = df['relevance_score'].mean()
    return f'Median relevance score: {med:.1f}/12 (mean {mean:.1f}). Distribution skews toward higher-quality matches.'

def insight_onet(df):
    flat = list(chain.from_iterable(df['onet_skills'].dropna()))
    if not flat: return 'No O*NET skills mapped under current filters.'
    top2 = [s for s, _ in Counter(flat).most_common(2)]
    both = df['onet_skills'].apply(lambda x: all(s in x for s in top2) if isinstance(x, list) else False).sum()
    pct  = int(both / max(len(df), 1) * 100)
    return (f'"{top2[0]}" and "{top2[1]}" co-occur in {pct}% of postings '
            '— a consistent dual requirement across the filtered dataset.')

def insight_trend(df):
    x, y = _trend_data(df)
    if len(y) < 2: return 'Insufficient date data for trend analysis.'
    recent_avg = round(sum(y[-4:]) / 4) if len(y) >= 4 else y[-1]
    growth = round((y[-1] - y[0]) / max(y[0], 1) * 100, 1)
    return f'Average of {recent_avg:,} postings ingested per week recently; {growth:+.1f}% cumulative growth over the window.'

def insight_skills(df):
    flat = list(chain.from_iterable(df['required_skills'].dropna()))
    if not flat: return 'No technical skills extracted under current filters.'
    top, cnt = Counter(flat).most_common(1)[0]
    pct = int(cnt / max(len(df), 1) * 100)
    return f'"{top}" appears in {pct}% of postings — the dominant technical skill under current filters.'

_INSIGHT_FNS = {
    'domain' : insight_domain,
    'tiers'  : insight_tiers,
    'source' : insight_source,
    'era'    : insight_era,
    'score'  : insight_score,
    'onet'   : insight_onet,
    'trend'  : insight_trend,
    'skills' : insight_skills,
}

for key in _INSIGHT_FNS:
    w = widgets.HTML(value=_style_insight(_INSIGHT_FNS[key](df_all)))
    _insight_widgets[key] = w

def _update_all_insights(df):
    for key, fn in _INSIGHT_FNS.items():
        if key in _insight_widgets:
            _insight_widgets[key].value = _style_insight(fn(df))

# ── Guided tour ───────────────────────────────────────────────────────────────
TOUR_STEPS = [
    {
        'title'  : 'Step 1 — Full dataset overview',
        'state'  : FilterState(),
        'caption': 'All domains, all tiers, all sources. Establishes the baseline volume and composition of the pipeline.'
    },
    {
        'title'  : 'Step 2 — AI domain deep dive',
        'state'  : FilterState(domain='AI'),
        'caption': 'AI postings only. Examine the O*NET competency profile and top technical skills unique to AI roles.'
    },
    {
        'title'  : 'Step 3 — Federal vs commercial split',
        'state'  : FilterState(domain='ALL', source='USAJobs'),
        'caption': 'USAJobs (federal) source only. Reveals the public-sector skill demand and GS-grade salary bands.'
    },
    {
        'title'  : 'Step 4 — HIGH-relevance postings only',
        'state'  : FilterState(tier=['HIGH'], score_min=6),
        'caption': 'Filters to the highest-quality postings (score ≥ 6). Most useful for curriculum alignment analysis.'
    },
    {
        'title'  : 'Step 5 — Current era postings',
        'state'  : FilterState(era='current'),
        'caption': 'Postings published ≥ Jan 1 2026. The most actionable slice for real-time workforce demand analysis.'
    },
]

_tour_step = [0]

_tour_title   = widgets.HTML(value='')
_tour_caption = widgets.HTML(value='')
_tour_prev    = widgets.Button(description='← Prev', layout=widgets.Layout(width='90px', height='32px'))
_tour_next    = widgets.Button(description='Next →', layout=widgets.Layout(width='90px', height='32px'))
_tour_prog    = widgets.HTML(value='')

def _tour_render():
    i    = _tour_step[0]
    step = TOUR_STEPS[i]
    _tour_title.value   = (f'<div style="font-size:13px;font-weight:500;color:{THEME["primary"]}">{step["title"]}</div>')
    _tour_caption.value = (f'<div style="font-size:12px;color:{THEME["muted"]};margin-top:4px">{step["caption"]}</div>')
    _tour_prog.value    = (f'<div style="font-size:11px;color:{THEME["muted"]}">{i+1} / {len(TOUR_STEPS)}</div>')
    # Apply tour state
    ts = step['state']
    state.domain    = ts.domain
    state.tier      = ts.tier
    state.source    = ts.source
    state.era       = ts.era
    state.score_min = ts.score_min
    state.score_max = ts.score_max
    state.keyword   = ts.keyword
    _apply_filters()

def _tour_go_prev(btn):
    if _tour_step[0] > 0:
        _tour_step[0] -= 1
        _tour_render()

def _tour_go_next(btn):
    if _tour_step[0] < len(TOUR_STEPS) - 1:
        _tour_step[0] += 1
        _tour_render()

_tour_prev.on_click(_tour_go_prev)
_tour_next.on_click(_tour_go_next)

tour_widget = widgets.VBox([
    widgets.HTML(value=f'<div style="font-size:13px;font-weight:500;color:{THEME["accent"]};margin-bottom:6px">📖 Guided Tour</div>'),
    widgets.HBox([_tour_prev, _tour_next, _tour_prog], layout=widgets.Layout(gap='8px', align_items='center')),
    _tour_title,
    _tour_caption,
],
    layout=widgets.Layout(
        border=f'1px solid {THEME["border"]}',
        border_radius='10px',
        padding='12px',
        background_color=THEME['surface'],
        margin='0 0 14px 0',
    )
)
_tour_render()
display(tour_widget)
print('Phase 6 ready — annotations + guided tour active.')


Phase 6 ready — annotations + guided tour active.


---
## Section 10G — Phase 7: Responsive Layout & Accessibility
A CSS Grid layout assembles all Phase 2–6 components into a three-zone dashboard: filter panel + KPI row at top, two-column chart grid in the middle, full-width O\*NET + skills panels at the bottom. ARIA attributes are injected via HTML.


In [40]:
# Phase 7 — Responsive CSS-Grid layout assembly

# ── Accessibility header ──────────────────────────────────────────────────────
display(HTML("""
<style>
  .db-header {
    font-family: Arial, sans-serif;
    border-bottom: 1px solid #E1E0D9;
    padding-bottom: 10px;
    margin-bottom: 14px;
  }
  .db-grid-2 {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(300px, 1fr));
    gap: 14px;
    margin-bottom: 14px;
  }
  .db-chart-card {
    background: #FFFFFF;
    border: 1px solid #E1E0D9;
    border-radius: 12px;
    padding: 14px;
    overflow: hidden;
  }
  .db-chart-card-wide {
    background: #FFFFFF;
    border: 1px solid #E1E0D9;
    border-radius: 12px;
    padding: 14px;
    margin-bottom: 14px;
  }
  /* Touch target sizing for all buttons */
  .widget-button, .widget-dropdown select {
    min-height: 36px !important;
  }
  /* Responsive: stack to 1 column on narrow viewports */
  @media (max-width: 600px) {
    .db-grid-2 {
      grid-template-columns: 1fr !important;
    }
  }
</style>
<h1 class="sr-only" style="position:absolute;width:1px;height:1px;overflow:hidden;clip:rect(0,0,0,0)">
  Job Pipeline Interactive Dashboard — CS, MBA, Data Analytics, AI
</h1>
"""))

# ── Dashboard header banner ───────────────────────────────────────────────────
_db_header = widgets.HTML(value=(
    '<div class="db-header">'
    f'<div style="font-size:18px;font-weight:500;color:{THEME["primary"]}">'
    'CS / MBA / Data Analytics / AI — Job Pipeline Dashboard</div>'
    f'<div style="font-size:12px;color:{THEME["muted"]};margin-top:3px">'
    'USAJobs + Jooble · O*NET 24.2 · Phase 1–9 Interactive Dashboard</div>'
    '</div>'
))

# ── Row 1: KPI output (already rendered above; wrap in output widget) ─────────
_kpi_section = widgets.VBox([
    widgets.HTML(value=f'<div style="font-size:12px;color:{THEME["muted"]};margin-bottom:6px">Key metrics</div>'),
    _P2_OUT,
])

# ── Row 2: Filter panel (already rendered) ────────────────────────────────────
# (filter_panel was displayed earlier; reference it here)

# ── Row 3: Tour + Top charts 2-column grid ────────────────────────────────────
_chart_grid_top = widgets.GridBox(
    [
        widgets.VBox([fig_dom,   _insight_widgets['domain']]),
        widgets.VBox([fig_tier,  _insight_widgets['tiers']]),
        widgets.VBox([fig_src,   _insight_widgets['source']]),
        widgets.VBox([fig_era,   _insight_widgets['era']]),
        widgets.VBox([fig_score, _insight_widgets['score']]),
        widgets.VBox([fig_trend, _insight_widgets['trend']]),
    ],
    layout=widgets.Layout(
        grid_template_columns='repeat(auto-fit, minmax(320px, 1fr))',
        gap='14px',
        width='100%',
    )
)

# ── Row 4: Full-width O*NET + Skills ─────────────────────────────────────────
_chart_wide_bot = widgets.GridBox(
    [
        widgets.VBox([fig_onet,   _insight_widgets['onet'],   _spotlight_out]),
        widgets.VBox([fig_skills, _insight_widgets['skills']]),
    ],
    layout=widgets.Layout(
        grid_template_columns='repeat(auto-fit, minmax(320px, 1fr))',
        gap='14px',
        width='100%',
    )
)

# ── Assemble full dashboard ───────────────────────────────────────────────────
full_dashboard = widgets.VBox([
    _db_header,
    tour_widget,
    filter_panel,
    _kpi_section,
    widgets.HTML(value=f'<div style="font-size:12px;color:{THEME["muted"]};margin:10px 0 4px">Charts</div>'),
    _chart_grid_top,
    widgets.HTML(value=f'<div style="font-size:12px;color:{THEME["muted"]};margin:10px 0 4px">Skills & O*NET</div>'),
    _chart_wide_bot,
],
    layout=widgets.Layout(width='100%', padding='4px')
)

display(full_dashboard)
print('Phase 7 ready — responsive dashboard assembled.')


Phase 7 ready — responsive dashboard assembled.


---
## Section 10H — Phase 8: Export Features
Three export pathways: (1) filtered CSV/XLSX/JSON download buttons, (2) a PDF summary report built with `reportlab` embedding chart images, (3) a base64 filter-state permalink for sharing exact dashboard views.


In [41]:
# Phase 8 — Export features
import base64, io, os
from datetime import datetime as _dt

_export_out = widgets.Output()

def _filtered_df():
    return state.apply(df_all)

# ── Helper: list-column flattener for CSV/XLSX ────────────────────────────────
def _flatten_lists(df):
    df2 = df.copy()
    for col in ['onet_skills', 'required_skills']:
        if col in df2.columns:
            df2[col] = df2[col].apply(
                lambda x: '|'.join(x) if isinstance(x, list) else str(x)
            )
    return df2

# ── Helper: create download link from bytes ───────────────────────────────────
def _download_link(data_bytes: bytes, filename: str, mime: str, label: str) -> str:
    b64 = base64.b64encode(data_bytes).decode()
    return (
        f'<a href="data:{mime};base64,{b64}" download="{filename}" '
        f'style="display:inline-block;padding:7px 16px;margin:4px;'
        f'background:{THEME["accent"]};color:#fff;border-radius:6px;'
        f'font-size:12px;text-decoration:none;font-family:Arial">{label}</a>'
    )

# ── Export: CSV ───────────────────────────────────────────────────────────────
def _export_csv(btn=None):
    df_f = _flatten_lists(_filtered_df())
    ts   = _dt.now().strftime('%Y%m%d_%H%M')
    buf  = io.StringIO(); df_f.to_csv(buf, index=False); buf.seek(0)
    data = buf.read().encode('utf-8-sig')
    with _export_out:
        clear_output(wait=True)
        display(HTML(_download_link(data, f'FILTERED_{ts}.csv', 'text/csv', '⬇ Download CSV')))

# ── Export: XLSX ──────────────────────────────────────────────────────────────
def _export_xlsx(btn=None):
    df_f = _flatten_lists(_filtered_df())
    ts   = _dt.now().strftime('%Y%m%d_%H%M')
    buf  = io.BytesIO()
    with pd.ExcelWriter(buf, engine='openpyxl') as xw:
        df_f.to_excel(xw, index=False, sheet_name='Filtered')
        # O*NET frequency sheet
        flat = list(chain.from_iterable(_filtered_df()['onet_skills'].dropna()))
        pd.DataFrame(Counter(flat).most_common(), columns=['onet_skill','frequency']).to_excel(
            xw, index=False, sheet_name='ONET_Frequency')
    buf.seek(0)
    mime = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
    with _export_out:
        clear_output(wait=True)
        display(HTML(_download_link(buf.read(), f'FILTERED_{ts}.xlsx', mime, '⬇ Download XLSX')))

# ── Export: JSON ──────────────────────────────────────────────────────────────
def _export_json(btn=None):
    df_f = _filtered_df()
    ts   = _dt.now().strftime('%Y%m%d_%H%M')
    payload = {
        'metadata': {
            'exported_at'   : _dt.utcnow().isoformat()+'Z',
            'filter_state'  : {
                'domain'    : state.domain,
                'tier'      : state.tier,
                'source'    : state.source,
                'era'       : state.era,
                'score_range': [state.score_min, state.score_max],
                'keyword'   : state.keyword,
            },
            'record_count'  : len(df_f),
        },
        'postings': df_f.to_dict(orient='records'),
    }
    data = json.dumps(payload, indent=2, default=str).encode('utf-8')
    with _export_out:
        clear_output(wait=True)
        display(HTML(_download_link(data, f'FILTERED_{ts}.json', 'application/json', '⬇ Download JSON')))

# ── Export: PDF summary report (reportlab) ────────────────────────────────────
def _export_pdf(btn=None):
    try:
        from reportlab.lib.pagesizes import letter
        from reportlab.lib import colors
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    except ImportError:
        with _export_out:
            display(HTML('<div style="color:red;font-size:12px">reportlab not installed — run: pip install reportlab</div>'))
        return

    df_f = _filtered_df()
    ts   = _dt.now().strftime('%Y%m%d_%H%M')
    buf  = io.BytesIO()
    doc  = SimpleDocTemplate(buf, pagesize=letter, topMargin=36, bottomMargin=36,
                             leftMargin=48, rightMargin=48)
    styles = getSampleStyleSheet()
    H1  = ParagraphStyle('H1', fontSize=16, spaceAfter=8, textColor=colors.HexColor('#1F3864'), fontName='Helvetica-Bold')
    H2  = ParagraphStyle('H2', fontSize=12, spaceAfter=6, textColor=colors.HexColor('#2E4057'), fontName='Helvetica-Bold')
    BDY = ParagraphStyle('BDY', fontSize=9, spaceAfter=4, leading=13)
    MUT = ParagraphStyle('MUT', fontSize=8, textColor=colors.HexColor('#898781'))

    flat_onet  = list(chain.from_iterable(df_f['onet_skills'].dropna()))
    top_onet   = Counter(flat_onet).most_common(10)
    flat_skills= list(chain.from_iterable(df_f['required_skills'].dropna()))
    top_skills = Counter(flat_skills).most_common(10)

    story = [
        Paragraph('Job Pipeline — Filtered Dataset Report', H1),
        Paragraph(f'Generated: {_dt.utcnow().strftime("%Y-%m-%d %H:%M UTC")} | Records: {len(df_f):,}', MUT),
        Spacer(1, 12),
        Paragraph('Filter state applied', H2),
        Paragraph(f'Domain: {state.domain}  |  Tier: {", ".join(state.tier)}  |  Source: {state.source}  |  Era: {state.era}  |  Score: {state.score_min}–{state.score_max}  |  Keyword: "{state.keyword}"', BDY),
        Spacer(1, 10),
        Paragraph('Key metrics', H2),
    ]
    kpi_table_data = [
        ['Metric', 'Value'],
        ['Total postings (filtered)', f'{len(df_f):,}'],
        ['Current era (≥ 2026)', f'{int((df_f["era"]=="current").sum()):,}'],
        ['HIGH relevance', f'{int((df_f["relevance_tier"]=="HIGH").sum()):,}'],
        ['Avg relevance score', f'{df_f["relevance_score"].mean():.2f} / 12'],
        ['USAJobs records', f'{int((df_f["source"]=="USAJobs").sum()):,}'],
        ['Jooble records', f'{int((df_f["source"]=="Jooble").sum()):,}'],
    ]
    kpi_t = Table(kpi_table_data, colWidths=[220, 150])
    kpi_t.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1F3864')),
        ('TEXTCOLOR',  (0,0), (-1,0), colors.white),
        ('FONTSIZE',   (0,0), (-1,-1), 8),
        ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#F2F5FA'), colors.white]),
        ('GRID', (0,0), (-1,-1), 0.25, colors.HexColor('#DDDDDD')),
        ('PADDING', (0,0), (-1,-1), 5),
    ]))
    story += [kpi_t, Spacer(1,10)]

    if top_onet:
        story.append(Paragraph('Top O*NET 24.2 competencies', H2))
        onet_data = [['Competency', '# Postings']] + [[s, str(n)] for s,n in top_onet]
        o_t = Table(onet_data, colWidths=[270, 100])
        o_t.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#264653')),
            ('TEXTCOLOR', (0,0), (-1,0), colors.white),
            ('FONTSIZE', (0,0), (-1,-1), 8),
            ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#F8F9FB'), colors.white]),
            ('GRID', (0,0), (-1,-1), 0.25, colors.HexColor('#DDDDDD')),
            ('PADDING', (0,0), (-1,-1), 4),
        ]))
        story += [o_t, Spacer(1,10)]

    if top_skills:
        story.append(Paragraph('Top technical skills', H2))
        sk_data = [['Skill', '# Postings']] + [[s, str(n)] for s,n in top_skills]
        sk_t = Table(sk_data, colWidths=[270, 100])
        sk_t.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#5B3A8C')),
            ('TEXTCOLOR', (0,0), (-1,0), colors.white),
            ('FONTSIZE', (0,0), (-1,-1), 8),
            ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor('#F8F9FB'), colors.white]),
            ('GRID', (0,0), (-1,-1), 0.25, colors.HexColor('#DDDDDD')),
            ('PADDING', (0,0), (-1,-1), 4),
        ]))
        story += [sk_t, Spacer(1,8)]

    story.append(Paragraph(
        'Methodology: Data collected via USAJobs API and Jooble API. Relevance scored 0–12 via title/description/org pattern matching. '
        'Competencies mapped to O*NET 24.2 Content Model. Deduplication via SHA-1 hash of title+company+location. '
        'Penn State Great Valley D.Eng. Research — AI in the Workforce & Labor Market.', MUT
    ))

    doc.build(story)
    buf.seek(0)
    with _export_out:
        clear_output(wait=True)
        display(HTML(_download_link(buf.read(), f'Report_{ts}.pdf', 'application/pdf', '⬇ Download PDF Report')))

# ── Filter state permalink ────────────────────────────────────────────────────
def _copy_permalink(btn=None):
    state_dict = {
        'd': state.domain, 't': state.tier, 's': state.source,
        'e': state.era, 'sm': state.score_min, 'sx': state.score_max, 'k': state.keyword
    }
    encoded = base64.urlsafe_b64encode(json.dumps(state_dict).encode()).decode()
    permalink = f'?state={encoded}'
    with _export_out:
        clear_output(wait=True)
        display(HTML(
            f'<div style="font-family:monospace;font-size:11px;background:#F2F5FA;'
            f'padding:8px 12px;border-radius:6px;border:1px solid #DDDDDD;'
            f'word-break:break-all;color:{THEME["primary"]}">{permalink}</div>'
            f'<div style="font-size:11px;color:{THEME["muted"]};margin-top:4px">'
            'Append this to your Voilà URL to restore this exact filter view.</div>'
        ))

# ── Export button row ─────────────────────────────────────────────────────────
_btn_style = {'button_color': THEME['accent'], 'font_color': '#FFFFFF'}
_btn_lay   = widgets.Layout(height='36px', width='150px')

_csv_btn  = widgets.Button(description='⬇ CSV',         button_style='', style=_btn_style, layout=_btn_lay)
_xlsx_btn = widgets.Button(description='⬇ XLSX',        button_style='', style=_btn_style, layout=_btn_lay)
_json_btn = widgets.Button(description='⬇ JSON',        button_style='', style=_btn_style, layout=_btn_lay)
_pdf_btn  = widgets.Button(description='⬇ PDF Report',  button_style='', style={'button_color':THEME['success'],'font_color':'#FFF'}, layout=_btn_lay)
_link_btn = widgets.Button(description='🔗 Permalink',   button_style='', style={'button_color':THEME['purple'],'font_color':'#FFF'},  layout=_btn_lay)

_csv_btn.on_click(_export_csv)
_xlsx_btn.on_click(_export_xlsx)
_json_btn.on_click(_export_json)
_pdf_btn.on_click(_export_pdf)
_link_btn.on_click(_copy_permalink)

export_panel = widgets.VBox([
    widgets.HTML(value=f'<div style="font-size:13px;font-weight:500;color:{THEME["primary"]};margin-bottom:8px">📤 Export current filtered view</div>'),
    widgets.HBox([_csv_btn, _xlsx_btn, _json_btn, _pdf_btn, _link_btn],
                 layout=widgets.Layout(flex_flow='row wrap', gap='8px')),
    _export_out,
],
    layout=widgets.Layout(
        border=f'1px solid {THEME["border"]}',
        border_radius='12px',
        padding='14px',
        background_color=THEME['surface'],
        margin='14px 0 0 0',
    )
)
display(export_panel)
print('Phase 8 ready — CSV / XLSX / JSON / PDF / Permalink export active.')


Phase 8 ready — CSV / XLSX / JSON / PDF / Permalink export active.


---
## Section 10I — Phase 9: GitHub Actions Auto-Deploy
This section provides the complete `weekly_job_collection.yml` workflow update, the `voila_config.json` for Voilà rendering, and a one-command static-HTML export. After merging these files, every Monday 6 AM UTC run will:
1. Execute the notebook via Papermill
2. Push master data files as a GitHub Release
3. Export the dashboard as a static HTML page
4. Deploy to GitHub Pages (`gh-pages` branch)


In [42]:
# Phase 9 — Write GitHub Actions workflow + Voilà config files
import json as _json, os as _os

# ── voila_config.json ─────────────────────────────────────────────────────────
VOILA_CONFIG = {
    "VoilaConfiguration": {
        "enable_nbextensions": True,
        "theme": "light",
        "strip_sources": True,
        "show_tracebacks": False
    }
}
voila_cfg_path = 'voila_config.json'
with open(voila_cfg_path, 'w') as f:
    _json.dump(VOILA_CONFIG, f, indent=2)
print(f'✅ {voila_cfg_path} written.')

# ── GitHub Actions workflow YAML ──────────────────────────────────────────────
WORKFLOW_YAML = '''
name: Weekly Job Pipeline + Dashboard Deploy

on:
  schedule:
    - cron: '0 6 * * 1'   # Every Monday at 06:00 UTC
  workflow_dispatch:       # Allow manual trigger from GitHub UI

permissions:
  contents: write
  id-token: write
  pages: write

jobs:
  # ── Job 1: Run notebook and release data ───────────────────────────────────
  run-notebook:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      # Authenticate via Workload Identity Federation (no stored key files)
      - id: auth
        uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: >-
            projects/71023542519/locations/global/workloadIdentityPools/
            github-pool/providers/github-provider-v2
          service_account: >-
            github-pipeline@project-bf62429c-050a-4ead-b92.iam.gserviceaccount.com

      - name: Set up Cloud SDK
        uses: google-github-actions/setup-gcloud@v2

      - name: Install Python dependencies
        run: |
          pip install papermill pandas openpyxl requests beautifulsoup4 tqdm \\
            jupyter nbconvert google-auth google-auth-oauthlib \\
            google-api-python-client plotly ipywidgets ipyvuetify \\
            kaleido reportlab voila

      - name: Execute notebook (data collection + dashboard)
        env:
          USAJOBS_EMAIL: ${{ secrets.USAJOBS_EMAIL }}
          USAJOBS_KEY:   ${{ secrets.USAJOBS_KEY }}
          JOOBLE_KEY:    ${{ secrets.JOOBLE_KEY }}
        run: |
          mkdir -p job_outputs
          papermill \\
            "CS_MBA_AI_USAJobs_Jooble_Master_Combiner_v3_(2).ipynb" \\
            ./executed_notebook.ipynb \\
            --log-output \\
            --no-progress-bar \\
            -p DRIVE_BASE job_outputs

      - name: Upload master files to Google Drive
        env:
          GDRIVE_FOLDER_ID: ${{ secrets.GDRIVE_FOLDER_ID }}
        run: python upload_to_drive.py

      - name: Create GitHub Release with master files
        uses: softprops/action-gh-release@v2
        with:
          tag_name: data-${{ github.run_id }}
          name: "Job Pipeline Data — Run ${{ github.run_id }}"
          body: |
            Automated weekly data release.
            - Source: USAJobs API + Jooble API
            - Competency framework: O*NET 24.2
            - Domains: CS, MBA, DATA_ANALYTICS, AI
          files: |
            job_outputs/outputs/MASTER_*.json
            job_outputs/outputs/MASTER_*.csv
            job_outputs/outputs/MASTER_*.xlsx
        env:
          GITHUB_TOKEN: ${{ secrets.GITHUB_TOKEN }}

      - name: Archive executed notebook artifact
        uses: actions/upload-artifact@v4
        if: always()
        with:
          name: executed-notebook-${{ github.run_id }}
          path: ./executed_notebook.ipynb

  # ── Job 2: Export Voilà dashboard → GitHub Pages ──────────────────────────
  deploy-dashboard:
    needs: [run-notebook]
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install rendering dependencies
        run: |
          pip install voila nbconvert plotly ipywidgets \\
            kaleido reportlab pandas openpyxl

      - name: Download executed notebook artifact
        uses: actions/download-artifact@v4
        with:
          name: executed-notebook-${{ github.run_id }}
          path: .

      - name: Rename executed notebook for export
        run: |
          mv executed_notebook.ipynb dashboard_notebook.ipynb

      - name: Export dashboard as static HTML via nbconvert
        run: |
          jupyter nbconvert \\
            --to html \\
            --execute \\
            --ExecutePreprocessor.timeout=600 \\
            --no-input \\
            dashboard_notebook.ipynb \\
            --output index.html

      - name: Deploy to GitHub Pages
        uses: peaceiris/actions-gh-pages@v4
        with:
          github_token: ${{ secrets.GITHUB_TOKEN }}
          publish_dir: ./
          publish_branch: gh-pages
          include_files: index.html
          commit_message: "Deploy dashboard — run ${{ github.run_id }}"
'''

workflow_dir  = '.github/workflows'
workflow_path = f'{workflow_dir}/weekly_job_collection.yml'
_os.makedirs(workflow_dir, exist_ok=True)
with open(workflow_path, 'w') as f:
    f.write(WORKFLOW_YAML.lstrip())
print(f'✅ {workflow_path} written.')

# ── Local Voilà launch helper ─────────────────────────────────────────────────
VOILA_LAUNCH = '''
#!/usr/bin/env bash
# Run this script from the repo root to serve the dashboard locally
# Usage: bash launch_dashboard.sh
set -e
NOTEBOOK="CS_MBA_AI_USAJobs_Jooble_Master_Combiner_v3_(2).ipynb"
echo "Starting Voilà dashboard on http://localhost:8866"
voila "$NOTEBOOK" \
  --no-browser=False \
  --port=8866 \
  --theme=light \
  --strip_sources=True
'''
with open('launch_dashboard.sh', 'w') as f:
    f.write(VOILA_LAUNCH.lstrip())
print('✅ launch_dashboard.sh written.')

# ── Inline static export for quick sharing ────────────────────────────────────
def export_html_snapshot():
    """Export the current notebook output as a standalone HTML file."""
    try:
        import subprocess, sys
        nb_path = 'CS_MBA_AI_USAJobs_Jooble_Master_Combiner_v3_(2).ipynb'
        out_path = 'dashboard_snapshot.html'
        result = subprocess.run(
            [sys.executable, '-m', 'nbconvert', '--to', 'html',
             '--no-input', nb_path, '--output', out_path],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode == 0:
            size_kb = _os.path.getsize(out_path) / 1024
            print(f'✅ HTML snapshot: {out_path} ({size_kb:.0f} KB)')
        else:
            print(f'nbconvert error: {result.stderr[:200]}')
    except Exception as e:
        print(f'Export skipped in this environment: {e}')

export_html_snapshot()

print()
print('Phase 9 complete. Summary of files written:')
print('  voila_config.json              — Voilà server configuration')
print('  .github/workflows/weekly_job_collection.yml  — Updated Actions workflow')
print('  launch_dashboard.sh            — Local Voilà launch script')
print()
print('Next steps:')
print('  1. Commit all three files to your GitHub repo')
print('  2. Enable GitHub Pages (Settings → Pages → Source: gh-pages branch)')
print('  3. The pipeline will auto-deploy every Monday at 6 AM UTC')
print('  4. Access dashboard at: https://rpp5069-cpu.github.io/job-pipeline/')


✅ voila_config.json written.
✅ .github/workflows/weekly_job_collection.yml written.
✅ launch_dashboard.sh written.
nbconvert error: [NbConvertApp] WARNING | pattern 'CS_MBA_AI_USAJobs_Jooble_Master_Combiner_v3_(2).ipynb' matched no files


Phase 9 complete. Summary of files written:
  voila_config.json              — Voilà server configuration
  .github/workflows/weekly_job_collection.yml  — Updated Actions workflow
  launch_dashboard.sh            — Local Voilà launch script

Next steps:
  1. Commit all three files to your GitHub repo
  2. Enable GitHub Pages (Settings → Pages → Source: gh-pages branch)
  3. The pipeline will auto-deploy every Monday at 6 AM UTC
  4. Access dashboard at: https://rpp5069-cpu.github.io/job-pipeline/


---
## Section 10J — Launch Dashboard
Run this single cell to render the complete nine-phase dashboard in one output block.


In [43]:
# ── Launch all nine phases in sequence ───────────────────────────────────────
clear_output(wait=True)

_header = widgets.HTML(value=(
    f'<div style="border-bottom:2px solid {THEME["primary"]};padding-bottom:10px;margin-bottom:16px">'
    f'<div style="font-size:20px;font-weight:500;color:{THEME["primary"]}">'
    'CS / MBA / Data Analytics / AI — Job Pipeline Dashboard</div>'
    f'<div style="font-size:12px;color:{THEME["muted"]};margin-top:3px">'
    f'Penn State Great Valley · D.Eng. Research · O*NET 24.2 · {len(df_all):,} postings loaded'
    '</div></div>'
))

# Re-render KPI row fresh
_kpi_out = widgets.Output()
with _kpi_out:
    render_kpi_row(state.apply(df_all))

_full = widgets.VBox([
    _header,
    tour_widget,
    filter_panel,
    _kpi_out,
    _chart_grid_top,
    _chart_wide_bot,
    export_panel,
],
    layout=widgets.Layout(width='100%', padding='6px')
)
display(_full)
print(f'Dashboard live — {len(df_all):,} postings · 9 phases active')


Dashboard live — 6,881 postings · 9 phases active
